# Model Comparison: Final Model vs Baselines

Compares the hybrid multi-scale CNN (Final model) against Logistic Regression, SVM, Random Forest, and ResNet1D on the 3-class CTG task using the same grouped 10-fold CV.

## 1. Imports

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import gc
import itertools
import numpy as np
import pandas as pd
from pathlib import Path
import wfdb
import warnings
from typing import Dict, List, Tuple, Optional
from scipy.interpolate import interp1d
from scipy.stats import loguniform, uniform

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, f1_score, accuracy_score,
    balanced_accuracy_score, precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except ImportError:
    from sklearn.model_selection import GroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = False

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.layers import (
    Conv1D, SeparableConv1D, BatchNormalization, Activation,
    AveragePooling1D, Dropout, GlobalAveragePooling1D, Dense,
    Concatenate, Reshape, Multiply, Add,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, Callback,
)
from tensorflow.keras.utils import to_categorical

import matplotlib.pyplot as plt
import seaborn as sns

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)
warnings.filterwarnings('ignore')
print(f"TensorFlow {tf.__version__} | NumPy {np.__version__}")

## 2. Configuration

In [ ]:
RAW_DATASET  = Path('../data/raw_dataset')
STEP2_LABELS = Path('../ExpertAnnotations/step2_labels.csv')
STEP3_LABELS = Path('../ExpertAnnotations/step3_labels.csv')
OUTPUT_DIR   = Path('outputs/model_comparison')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_SAMPLING_RATE    = 4
TARGET_SAMPLING_RATE = 1
DOWNSAMPLE_FACTOR    = RAW_SAMPLING_RATE // TARGET_SAMPLING_RATE

WINDOW_MINUTES = 30
WINDOW_LENGTH  = WINDOW_MINUTES * 60 * TARGET_SAMPLING_RATE
SAMPLING_RATE  = TARGET_SAMPLING_RATE
SEGMENT_LENGTH = WINDOW_LENGTH

NUM_CLASSES = 3
CLASS_NAMES = ['Normal', 'Mild', 'Severe']
LABEL_MAP   = {1: 0, 2: 1, 3: 2}

CFG = dict(
    step_numbers             = [2, 3],
    mode_a_offsets           = {2: (60, 30), 3: (30, 0)},
    controlled_step_offsets  = {
        2: [(70, 40), (60, 30), (50, 20)],
        3: [(50, 20), (40, 10), (30, 0)],
    },
    segment_minutes          = WINDOW_MINUTES,
    segment_length           = WINDOW_LENGTH,
    max_nan_fraction         = 0.16,
    pad_short_records        = False,
    n_splits                 = 10,
    cv_seed                  = 42,
    min_severe_records       = 10,
    max_cv_attempts          = 100,
    temporal_filters         = 32,
    temporal_kernel          = 7,
    separable_filters        = 32,
    separable_kernels        = [3, 7, 15],
    projection_filters       = 32,
    dropout_rate             = 0.40,
    residual_dilation_rates  = [2, 4],
    use_clinical_features    = True,
    augment_train            = True,
    aug_time_warp            = True,
    aug_warp_sigma           = 0.10,
    aug_amplitude_scale      = True,
    aug_amp_sigma            = 0.10,
    aug_noise                = True,
    aug_noise_std            = 0.02,
    aug_signal_loss          = True,
    aug_gap_frac             = 0.05,
    aug_n_gaps               = 2,
    scratch_epochs           = 100,
    scratch_lr               = 1e-3,
    scratch_batch            = 16,
    scratch_patience         = 20,
    label_smoothing          = 0.05,
    use_class_weights        = True,
)

windows_per_step = {s: len(v) for s, v in CFG['controlled_step_offsets'].items()}
CONTROLLED_WINDOWS_PER_STEP = next(iter(windows_per_step.values()))
print(f'CONTROLLED_WINDOWS_PER_STEP = {CONTROLLED_WINDOWS_PER_STEP}')

## 3. Data Loading

In [ ]:
def load_step_labels(step_paths: Dict[int, Path]) -> Dict[int, pd.DataFrame]:
    step_labels = {}
    for step_num, path in step_paths.items():
        df = pd.read_csv(path)
        df = df[df['Majority_Vote_Label'] != -1].copy()
        df['class_id'] = df['Majority_Vote_Label'].map(LABEL_MAP)
        df['rec_id'] = df['rec_id'].astype(str)
        step_labels[step_num] = df
        print(f'Step {step_num}: {len(df)} labelled records')
    return step_labels

In [ ]:
def remove_trailing_zeros(signal) -> list:
    sig = list(signal) if isinstance(signal, np.ndarray) else list(signal)
    i = len(sig) - 1
    while i >= 0 and sig[i] == 0:
        i -= 1
    return sig[:i + 1]

In [ ]:
def clean_fhr(fhr_array, fs: int = 4) -> np.ndarray:
    fhr = pd.Series(np.array(fhr_array, dtype=float))
    fhr.replace(0, np.nan, inplace=True)
    na = fhr.isnull()
    gap_groups = na.ne(na.shift()).cumsum()
    gap_sizes = fhr.groupby(gap_groups.values).transform('size')
    fhr = fhr[~(gap_sizes.ge(fs * 15 + 1) & na)].reset_index(drop=True)
    fhr[fhr < 50] = np.nan
    fhr[fhr > 200] = np.nan
    fhr = fhr.interpolate(method='linear')
    diff = fhr - fhr.shift()
    fhr[(diff > 25) | (diff < -25)] = np.nan
    fhr = fhr.interpolate(method='linear')
    fhr = fhr.ffill().bfill()
    return fhr.values

In [ ]:
def downsample_signal(signal: np.ndarray, factor: int = 4) -> np.ndarray:
    if factor <= 1:
        return np.asarray(signal, dtype=np.float32)
    return np.asarray(signal[::factor], dtype=np.float32)

In [ ]:
def load_raw_signals(dataset_path: Path) -> Dict[str, dict]:
    records = [p.stem for p in dataset_path.glob('*.hea')]
    data = {}
    for rid in sorted(records):
        try:
            rec = wfdb.rdrecord(str(dataset_path / rid))
            fhr_raw_4hz = np.asarray(remove_trailing_zeros(rec.p_signal[:, 0].tolist()), dtype=np.float32)
            if fhr_raw_4hz.size == 0:
                raise ValueError('empty FHR after trimming trailing zeros')
            fhr_clean_4hz = np.asarray(clean_fhr(fhr_raw_4hz, fs=RAW_SAMPLING_RATE), dtype=np.float32)
            fhr_clean_1hz = downsample_signal(fhr_clean_4hz, factor=DOWNSAMPLE_FACTOR)
            data[rid] = {'FHR': fhr_clean_1hz, 'length': len(fhr_clean_1hz)}
        except Exception as e:
            print(f'  Skipping {rid}: {e}')
    print(f'Loaded {len(data)} FHR signals')
    return data

In [ ]:
STEP_LABEL_PATHS = {2: STEP2_LABELS, 3: STEP3_LABELS}
step_label_dfs = load_step_labels(STEP_LABEL_PATHS)
signal_data    = load_raw_signals(RAW_DATASET)

labelled_rec_ids = set()
for df in step_label_dfs.values():
    labelled_rec_ids.update(df['rec_id'].tolist())
valid_rec_ids = sorted(labelled_rec_ids & set(signal_data.keys()))
print(f'Records with label and signal: {len(valid_rec_ids)}')

## 4. Window Extraction

In [ ]:
def _window_bounds_from_end(
    sig_len: int,
    offsets_from_end_min: Tuple[int, int],
    fs: int = 1,
) -> Optional[Tuple[int, int]]:
    start_from_end_min, end_from_end_min = offsets_from_end_min
    start_from_end = start_from_end_min * 60 * fs
    end_from_end   = end_from_end_min * 60 * fs
    if sig_len < start_from_end:
        return None
    start_idx = sig_len - start_from_end
    end_idx   = sig_len - end_from_end if end_from_end > 0 else sig_len
    if end_idx <= start_idx:
        return None
    return int(start_idx), int(end_idx)

In [ ]:
def _get_controlled_window_bounds(
    sig_len: int,
    step_num: int,
    controlled_step_offsets: Dict[int, List[Tuple[int, int]]],
    fs: int = 1,
) -> Optional[List[Tuple[int, Tuple[int, int], int, int]]]:
    bounds = []
    for window_index, offsets in enumerate(controlled_step_offsets[step_num]):
        cur = _window_bounds_from_end(sig_len, offsets, fs=fs)
        if cur is None:
            return None
        bounds.append((window_index, offsets, cur[0], cur[1]))
    return bounds

In [ ]:
def create_step_windows(
    signal_data: Dict[str, dict],
    step_label_dfs: Dict[int, pd.DataFrame],
    record_ids: List[str],
    window_length: int = SEGMENT_LENGTH,
    step_numbers: Optional[List[int]] = None,
    controlled_step_offsets: Optional[Dict[int, List[Tuple[int, int]]]] = None,
    max_nan_fraction: float = 0.16,
    pad_short_records: bool = False,
    fs: int = 1,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    if step_numbers is None:
        step_numbers = [2, 3]
    if controlled_step_offsets is None:
        controlled_step_offsets = CFG['controlled_step_offsets']
    fixed_wps = len(next(iter(controlled_step_offsets.values())))

    X_list, y_list, group_list, meta_list = [], [], [], []
    accepted = skipped_short = skipped_nan = 0

    for rid in record_ids:
        if rid not in signal_data:
            continue
        fhr = np.array(signal_data[rid]['FHR'], dtype=np.float64)
        sig_len = len(fhr)
        for step_num in step_numbers:
            step_df = step_label_dfs.get(step_num)
            if step_df is None:
                continue
            row = step_df[step_df['rec_id'] == rid]
            if row.empty:
                continue
            controlled_bounds = _get_controlled_window_bounds(
                sig_len, step_num, controlled_step_offsets, fs=fs
            )
            if controlled_bounds is None:
                skipped_short += 1
                continue
            label = int(row['class_id'].iloc[0])
            step_windows = []
            failed = None
            for window_index, offsets, start_idx, end_idx in controlled_bounds:
                window = fhr[start_idx:end_idx]
                if len(window) < window_length:
                    if pad_short_records:
                        window = np.pad(window, (window_length - len(window), 0), mode='edge')
                    else:
                        failed = 'short'
                        break
                missing_frac = float(np.mean(np.isnan(window) | (window == 0)))
                if missing_frac > max_nan_fraction:
                    failed = 'nan'
                    break
                if np.any(np.isnan(window)):
                    s = pd.Series(window)
                    window = s.interpolate(method='linear').ffill().bfill().values
                step_windows.append({
                    'window': window.astype(np.float32),
                    'window_index': window_index,
                    'offset_start_min': int(offsets[0]),
                    'offset_end_min': int(offsets[1]),
                    'missing_frac_prefill': missing_frac,
                })
            if failed == 'short':
                skipped_short += 1
                continue
            if failed == 'nan':
                skipped_nan += 1
                continue
            accepted += 1
            for sw in step_windows:
                X_list.append(sw['window'])
                y_list.append(label)
                group_list.append(rid)
                meta_list.append({
                    'rec_id': rid, 'step': step_num,
                    'window_index': sw['window_index'],
                    'label': label,
                    'missing_frac_prefill': sw['missing_frac_prefill'],
                })

    X = np.array(X_list, dtype=np.float32)[:, :, np.newaxis] if X_list else np.empty((0, window_length, 1), dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    groups = np.array(group_list)
    metadata = pd.DataFrame(meta_list)
    print(f'Windows: {len(X)}, records: {len(set(group_list))}, accepted steps: {accepted}')
    return X, y, groups, metadata

In [ ]:
X_all, y_all, groups_all, meta_all = create_step_windows(
    signal_data=signal_data,
    step_label_dfs=step_label_dfs,
    record_ids=valid_rec_ids,
    window_length=CFG['segment_length'],
    step_numbers=CFG['step_numbers'],
    controlled_step_offsets=CFG['controlled_step_offsets'],
    max_nan_fraction=CFG['max_nan_fraction'],
    pad_short_records=CFG['pad_short_records'],
    fs=SAMPLING_RATE,
)
print(f'X_all: {X_all.shape}, y_all: {y_all.shape}')

## 5. Normalization and Augmentation

In [ ]:
def compute_norm_stats(X_train: np.ndarray) -> dict:
    vals = X_train.ravel()
    mean = float(np.mean(vals))
    std  = float(np.std(vals))
    if std < 1e-8:
        std = 1.0
    return {'mean': mean, 'std': std}

In [ ]:
def apply_norm(X: np.ndarray, stats: dict) -> np.ndarray:
    X_norm  = X.astype(np.float32)
    X_norm -= stats['mean']
    X_norm /= stats['std']
    return X_norm

In [ ]:
def time_warp(X: np.ndarray, sigma: float = 0.1) -> np.ndarray:
    N, T, C = X.shape
    X_warped  = np.empty_like(X)
    orig_steps = np.arange(T)
    knot_pos   = np.linspace(0, T - 1, 6)
    for i in range(N):
        warp_factors    = np.random.normal(1.0, sigma, size=6)
        warp_factors[0] = 1.0
        warp_factors[-1]= 1.0
        warped_knots    = np.cumsum(np.diff(knot_pos) * warp_factors[:-1])
        warped_knots    = np.concatenate([[0], warped_knots])
        warped_knots    = warped_knots / warped_knots[-1] * (T - 1)
        interp_fn       = interp1d(warped_knots, knot_pos, kind='linear', fill_value='extrapolate')
        warped_steps    = np.clip(interp_fn(orig_steps), 0, T - 1)
        for c in range(C):
            interp_sig      = interp1d(orig_steps, X[i, :, c], kind='linear', fill_value='extrapolate')
            X_warped[i,:,c] = interp_sig(warped_steps)
    return X_warped.astype(np.float32)

In [ ]:
def amplitude_scale(X: np.ndarray, sigma: float = 0.1) -> np.ndarray:
    scales = np.random.normal(1.0, sigma, size=(X.shape[0], 1, 1)).astype(np.float32)
    return X * scales

In [ ]:
def additive_noise(X: np.ndarray, noise_std: float = 0.02) -> np.ndarray:
    return X + np.random.normal(0, noise_std, X.shape).astype(np.float32)

In [ ]:
def signal_loss_simulation(X: np.ndarray, max_gap_frac: float = 0.05, n_gaps: int = 2) -> np.ndarray:
    N, T, C = X.shape
    X_aug   = X.copy()
    max_gap = max(1, int(T * max_gap_frac))
    for i in range(N):
        for _ in range(n_gaps):
            gap_len = np.random.randint(1, max_gap + 1)
            start   = np.random.randint(0, T - gap_len)
            end     = start + gap_len
            for c in range(C):
                left  = X_aug[i, max(start - 1, 0), c]
                right = X_aug[i, min(end, T - 1), c]
                X_aug[i, start:end, c] = np.linspace(left, right, gap_len)
    return X_aug.astype(np.float32)

In [ ]:
def augment_signal(X: np.ndarray, cfg: dict) -> np.ndarray:
    X_aug = X.copy()
    if cfg.get('aug_time_warp', True):
        X_aug = time_warp(X_aug, sigma=cfg.get('aug_warp_sigma', 0.1))
    if cfg.get('aug_amplitude_scale', True):
        X_aug = amplitude_scale(X_aug, sigma=cfg.get('aug_amp_sigma', 0.1))
    if cfg.get('aug_noise', True):
        X_aug = additive_noise(X_aug, noise_std=cfg.get('aug_noise_std', 0.02))
    if cfg.get('aug_signal_loss', True):
        X_aug = signal_loss_simulation(
            X_aug, max_gap_frac=cfg.get('aug_gap_frac', 0.05), n_gaps=cfg.get('aug_n_gaps', 2)
        )
    return X_aug

## 6. Engineered Features

In [ ]:
def _count_episodes(deviation: np.ndarray, mask: np.ndarray, min_dur: int) -> tuple:
    count = 0
    total_area = 0.0
    in_episode = False
    episode_start = 0
    for j in range(len(mask)):
        if mask[j] and not in_episode:
            in_episode = True
            episode_start = j
        elif not mask[j] and in_episode:
            in_episode = False
            dur = j - episode_start
            if dur >= min_dur:
                count += 1
                total_area += float(np.sum(np.abs(deviation[episode_start:j])))
    if in_episode:
        dur = len(mask) - episode_start
        if dur >= min_dur:
            count += 1
            total_area += float(np.sum(np.abs(deviation[episode_start:])))
    return count, total_area

In [ ]:
def compute_clinical_features(
    X: np.ndarray,
    fs: int = 4,
    missing_frac_prefill=None,
) -> np.ndarray:
    N, T, _ = X.shape
    features = np.zeros((N, 10), dtype=np.float32)
    min_dur_samples = 15 * fs
    one_min_samples = 60 * fs
    if missing_frac_prefill is None:
        missing_frac_prefill = np.zeros(N, dtype=np.float32)
    else:
        missing_frac_prefill = np.asarray(missing_frac_prefill, dtype=np.float32)
    for i in range(N):
        sig = np.asarray(X[i, :, 0], dtype=np.float64)
        baseline  = float(np.median(sig))
        deviation = sig - baseline
        features[i, 0] = float(np.mean(sig))
        features[i, 1] = float(np.median(sig))
        features[i, 2] = float(np.std(sig))
        q25, q75 = np.percentile(sig, [25, 75])
        features[i, 3] = float(q75 - q25)
        features[i, 4] = float(np.mean(np.abs(np.diff(sig)))) if len(sig) > 1 else 0.0
        n_minutes = T // one_min_samples
        if n_minutes >= 2:
            minute_means = [
                np.mean(sig[j * one_min_samples:(j + 1) * one_min_samples])
                for j in range(n_minutes)
            ]
            features[i, 5] = float(np.std(minute_means))
        else:
            features[i, 5] = features[i, 2]
        accel_mask = deviation >= 15
        decel_mask = deviation <= -15
        accel_count, _ = _count_episodes(deviation, accel_mask, min_dur_samples)
        decel_count, _ = _count_episodes(-deviation, decel_mask, min_dur_samples)
        features[i, 6] = float(accel_count)
        features[i, 7] = float(decel_count)
        features[i, 8] = float(np.max(np.maximum(-deviation, 0.0)))
        features[i, 9] = float(missing_frac_prefill[i])
    return features

In [ ]:
CLINICAL_FEATURE_NAMES = [
    'mean_fhr', 'median_fhr', 'std_fhr', 'iqr_fhr', 'masd',
    'minute_mean_std', 'n_accel', 'n_decel', 'max_decel_depth', 'missing_frac_prefill',
]
N_CLINICAL_FEATURES = len(CLINICAL_FEATURE_NAMES)
print(f'{N_CLINICAL_FEATURES} clinical features: {CLINICAL_FEATURE_NAMES}')

In [ ]:
def normalize_clinical_features(feats_train: np.ndarray, feats_apply: np.ndarray) -> tuple:
    mean = feats_train.mean(axis=0)
    std  = feats_train.std(axis=0)
    std[std < 1e-8] = 1.0
    return (feats_apply - mean) / std, {'mean': mean, 'std': std}

## 7. Model Architecture

In [ ]:
def _se_block_1d(x, ratio: int = 4, name: str = 'se'):
    filters = int(x.shape[-1])
    se = GlobalAveragePooling1D(name=f'{name}_gap')(x)
    se = Dense(max(filters // ratio, 4), activation='elu', name=f'{name}_fc1')(se)
    se = Dense(filters, activation='sigmoid', name=f'{name}_fc2')(se)
    se = Reshape((1, filters), name=f'{name}_reshape')(se)
    return Multiply(name=f'{name}_scale')([x, se])

In [ ]:
def build_model(
    input_length: int = SEGMENT_LENGTH,
    num_classes: int = 3,
    temporal_filters: int = 32,
    temporal_kernel: int = 7,
    separable_filters: int = 32,
    separable_kernels: List[int] = None,
    projection_filters: int = 32,
    dropout_rate: float = 0.40,
    n_clinical_features: int = 0,
    residual_dilation_rates: List[int] = None,
) -> Model:
    if separable_kernels is None:
        separable_kernels = [3, 7, 15]
    if residual_dilation_rates is None:
        residual_dilation_rates = [2, 4]

    inputs = Input(shape=(input_length, 1), name='fhr_input')
    x = Conv1D(temporal_filters, temporal_kernel, padding='same', use_bias=False, name='temporal_conv')(inputs)
    x = BatchNormalization(name='temporal_bn')(x)
    x = Activation('elu', name='temporal_elu')(x)
    x = AveragePooling1D(pool_size=4, name='pool1')(x)
    x = Dropout(dropout_rate, name='dropout1')(x)

    if len(separable_kernels) > 1:
        branches = []
        for i, k in enumerate(separable_kernels):
            branch = SeparableConv1D(separable_filters, k, padding='same', use_bias=False, name=f'ms_sep_{i}')(x)
            branch = BatchNormalization(name=f'ms_bn_{i}')(branch)
            branch = Activation('elu', name=f'ms_elu_{i}')(branch)
            branches.append(branch)
        x = Concatenate(axis=-1, name='ms_concat')(branches)
    else:
        x = SeparableConv1D(separable_filters, separable_kernels[0], padding='same', use_bias=False, name='ms_sep_0')(x)
        x = BatchNormalization(name='ms_bn_0')(x)
        x = Activation('elu', name='ms_elu_0')(x)

    x = Conv1D(projection_filters, 1, use_bias=False, name='proj1')(x)
    x = BatchNormalization(name='proj1_bn')(x)
    x = Activation('elu', name='proj1_elu')(x)
    x = _se_block_1d(x, ratio=4, name='se1')
    x = AveragePooling1D(pool_size=4, name='pool2')(x)
    x = Dropout(dropout_rate, name='dropout2')(x)

    residual = x
    dilation_rates = list(residual_dilation_rates)
    if len(dilation_rates) < 2:
        dilation_rates = dilation_rates + [dilation_rates[-1]]
    x = SeparableConv1D(projection_filters, 7, padding='same', dilation_rate=dilation_rates[0], use_bias=False, name='res_sep1')(x)
    x = BatchNormalization(name='res_bn1')(x)
    x = Activation('elu', name='res_elu1')(x)
    x = SeparableConv1D(projection_filters, 7, padding='same', dilation_rate=dilation_rates[1], use_bias=False, name='res_sep2')(x)
    x = BatchNormalization(name='res_bn2')(x)
    x = Add(name='res_add')([x, residual])
    x = Activation('elu', name='res_elu2')(x)
    x = _se_block_1d(x, ratio=4, name='se2')
    x = AveragePooling1D(pool_size=2, name='pool3')(x)
    x = Dropout(dropout_rate, name='dropout3')(x)
    x = GlobalAveragePooling1D(name='global_pool')(x)

    if n_clinical_features > 0:
        clinical_input = Input(shape=(n_clinical_features,), name='clinical_input')
        clin = Dense(16, activation='elu', name='clinical_fc')(clinical_input)
        clin = Dropout(0.2, name='clinical_drop')(clin)
        x = Concatenate(name='merge_clin')([x, clin])
        x = Dense(32, activation='elu', name='fusion_fc')(x)
        x = Dropout(dropout_rate, name='fusion_drop')(x)
        outputs = Dense(num_classes, activation='softmax', name='output')(x)
        return Model([inputs, clinical_input], outputs, name='FHR-SegNet-ClinFeat')

    outputs = Dense(num_classes, activation='softmax', name='output')(x)
    return Model(inputs, outputs, name='FHR-SegNet')

## 8. ResNet1D Architecture

In [ ]:
def _resnet1d_block(x, filters: int, kernel_size: int = 7, name: str = 'res'):
    shortcut = x
    if int(x.shape[-1]) != filters:
        shortcut = Conv1D(filters, 1, padding='same', use_bias=False, name=f'{name}_proj')(shortcut)
        shortcut = BatchNormalization(name=f'{name}_proj_bn')(shortcut)
    x = Conv1D(filters, kernel_size, padding='same', use_bias=False, name=f'{name}_conv1')(x)
    x = BatchNormalization(name=f'{name}_bn1')(x)
    x = Activation('relu', name=f'{name}_relu1')(x)
    x = Conv1D(filters, kernel_size, padding='same', use_bias=False, name=f'{name}_conv2')(x)
    x = BatchNormalization(name=f'{name}_bn2')(x)
    x = Add(name=f'{name}_add')([x, shortcut])
    x = Activation('relu', name=f'{name}_relu2')(x)
    return x

In [ ]:
def build_resnet1d(
    input_length: int = SEGMENT_LENGTH,
    num_classes: int = NUM_CLASSES,
    dropout_rate: float = 0.40,
    kernel_size: int = 7,
    base_filters: int = 64,
) -> Model:
    inputs = Input(shape=(input_length, 1), name='resnet1d_input')
    x = Conv1D(base_filters, kernel_size, padding='same', use_bias=False, name='resnet1d_init_conv')(inputs)
    x = BatchNormalization(name='resnet1d_init_bn')(x)
    x = Activation('relu', name='resnet1d_init_relu')(x)
    x = layers.MaxPooling1D(pool_size=4, name='resnet1d_init_pool')(x)
    x = _resnet1d_block(x, filters=base_filters, kernel_size=kernel_size, name='resnet1d_blk1')
    x = layers.MaxPooling1D(pool_size=4, name='resnet1d_pool1')(x)
    x = _resnet1d_block(x, filters=base_filters * 2, kernel_size=kernel_size, name='resnet1d_blk2')
    x = layers.MaxPooling1D(pool_size=4, name='resnet1d_pool2')(x)
    x = _resnet1d_block(x, filters=base_filters * 2, kernel_size=kernel_size, name='resnet1d_blk3')
    x = GlobalAveragePooling1D(name='resnet1d_gap')(x)
    x = Dropout(dropout_rate, name='resnet1d_drop')(x)
    outputs = Dense(num_classes, activation='softmax', name='resnet1d_output')(x)
    return Model(inputs, outputs, name='ResNet1D')

## 9. Grouped CV Utilities

In [ ]:
def _ensure_step_keys(metadata: pd.DataFrame) -> pd.DataFrame:
    meta = metadata.copy()
    if 'step_key' not in meta.columns:
        meta['step_key'] = meta['rec_id'].astype(str) + '__step' + meta['step'].astype(str)
    return meta

In [ ]:
def build_step_target_table(metadata: pd.DataFrame) -> pd.DataFrame:
    meta = _ensure_step_keys(metadata)
    step_targets = meta[['step_key', 'rec_id', 'step', 'label']].drop_duplicates().copy()
    return step_targets.sort_values(['rec_id', 'step']).reset_index(drop=True)

In [ ]:
def stratified_record_kfold_with_constraint(
    metadata: pd.DataFrame,
    n_splits: int = 10,
    min_severe_records: int = 10,
    max_attempts: int = 100,
    cv_seed: int = 42,
    severe_class: int = 2,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    step_targets = build_step_target_table(meta)
    step_y      = step_targets['label'].to_numpy(dtype=int)
    step_groups = step_targets['rec_id'].to_numpy()
    step_keys   = step_targets['step_key'].to_numpy()

    total_severe  = int(np.sum(step_y == severe_class))
    min_possible  = total_severe // n_splits if n_splits > 0 else 0
    effective_min = min(min_severe_records, max(min_possible - 1, 1)) if total_severe > 0 else 0
    print(f'  Total Severe step-units: {total_severe}, effective_min per fold: {effective_min}')

    best_folds = None
    best_min_severe = -1

    n_attempts = max_attempts if HAS_STRATIFIED_GROUP_KFOLD else 1
    for attempt in range(n_attempts):
        seed = cv_seed + attempt
        if HAS_STRATIFIED_GROUP_KFOLD:
            splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        else:
            from sklearn.model_selection import GroupKFold
            splitter = GroupKFold(n_splits=n_splits)
        candidate_folds = []
        fold_severe_counts = []
        all_ok = True
        for train_step_idx, val_step_idx in splitter.split(step_keys, step_y, step_groups):
            train_rec_ids = set(step_targets.iloc[train_step_idx]['rec_id'])
            val_rec_ids   = set(step_targets.iloc[val_step_idx]['rec_id'])
            n_sev = int(np.sum(step_y[val_step_idx] == severe_class))
            fold_severe_counts.append(n_sev)
            train_mask = meta['rec_id'].isin(train_rec_ids).to_numpy()
            val_mask   = meta['rec_id'].isin(val_rec_ids).to_numpy()
            candidate_folds.append((np.where(train_mask)[0], np.where(val_mask)[0]))
            if n_sev < effective_min:
                all_ok = False
        cur_min = min(fold_severe_counts) if fold_severe_counts else -1
        if cur_min > best_min_severe:
            best_min_severe = cur_min
            best_folds = candidate_folds
        if all_ok:
            print(f'  Found valid split at attempt {attempt + 1} (seed={seed})')
            print(f'    Severe per val fold: {fold_severe_counts}')
            return candidate_folds

    print(f'  Best found: min={best_min_severe}')
    return best_folds

## 10. Evaluation Utilities

In [ ]:
def aggregate_to_record_step_level(
    y_proba: np.ndarray,
    metadata: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    unique_step_keys = meta['step_key'].unique()
    step_y_true, step_y_pred = [], []
    for step_key in unique_step_keys:
        mask = meta['step_key'] == step_key
        mean_proba = y_proba[mask].mean(axis=0)
        step_y_pred.append(int(np.argmax(mean_proba)))
        step_y_true.append(int(meta.loc[mask, 'label'].iloc[0]))
    return (
        unique_step_keys,
        np.array(step_y_true, dtype=int),
        np.array(step_y_pred, dtype=int),
    )

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    labels = list(range(NUM_CLASSES))
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1= f1_score(y_true, y_pred, average='macro', labels=labels, zero_division=0)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    metrics = {'accuracy': acc, 'balanced_accuracy': bal_acc, 'macro_f1': macro_f1, 'confusion_matrix': cm}
    for i, name in enumerate(CLASS_NAMES):
        metrics[f'precision_{name}'] = precision[i]
        metrics[f'recall_{name}']    = recall[i]
        metrics[f'f1_{name}']        = f1[i]
        metrics[f'support_{name}']   = int(support[i]) if support[i] is not None else 0
    return metrics

In [ ]:
def summarise_fold_results(fold_results: List[dict], model_name: str) -> dict:
    summary = {'model': model_name}
    for key in ['accuracy', 'balanced_accuracy', 'macro_f1']:
        vals = [r[key] for r in fold_results]
        summary[f'{key}_mean'] = float(np.mean(vals))
        summary[f'{key}_std']  = float(np.std(vals))
    for name in CLASS_NAMES:
        vals = [r[f'recall_{name}'] for r in fold_results]
        summary[f'{name.lower()}_recall_mean'] = float(np.mean(vals))
        summary[f'{name.lower()}_recall_std']  = float(np.std(vals))
    return summary

## 11. Keras Training Callback

In [ ]:
class RecordStepBalancedAccuracy(Callback):
    def __init__(self, X_val, y_val, windows_per_step=CONTROLLED_WINDOWS_PER_STEP, batch_size=32, verbose=True):
        super().__init__()
        self.X_val = X_val
        self.y_val = np.asarray(y_val, dtype=np.int32)
        self.windows_per_step = int(windows_per_step)
        self.batch_size = batch_size
        self.verbose = bool(verbose)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_proba = self.model.predict(self.X_val, batch_size=self.batch_size, verbose=0)
        step_y_true, step_y_pred = [], []
        for start in range(0, len(y_proba), self.windows_per_step):
            end = start + self.windows_per_step
            mean_proba = y_proba[start:end].mean(axis=0)
            step_y_true.append(int(self.y_val[start]))
            step_y_pred.append(int(np.argmax(mean_proba)))
        bal_acc = float(balanced_accuracy_score(step_y_true, step_y_pred))
        logs['val_record_step_balanced_accuracy'] = bal_acc
        if self.verbose:
            print(f'  val_record_step_balanced_accuracy: {bal_acc:.4f}')

In [ ]:
def get_callbacks(
    patience: int = 20,
    monitor: str = 'val_record_step_balanced_accuracy',
    monitor_mode: str = 'max',
    prefix_callbacks=None,
) -> list:
    callbacks = list(prefix_callbacks or [])
    callbacks.extend([
        EarlyStopping(monitor=monitor, mode=monitor_mode, patience=patience, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor=monitor, mode=monitor_mode, factor=0.5, patience=max(patience // 2, 3), min_lr=1e-6, verbose=1),
    ])
    return callbacks

## 12. Shared CV Folds and Clinical Features

In [ ]:
baseline_folds = stratified_record_kfold_with_constraint(
    meta_all,
    n_splits=CFG['n_splits'],
    min_severe_records=CFG['min_severe_records'],
    max_attempts=CFG['max_cv_attempts'],
    cv_seed=CFG['cv_seed'],
)
print(f'Number of CV folds: {len(baseline_folds)}')

clin_all = compute_clinical_features(
    X_all,
    fs=SAMPLING_RATE,
    missing_frac_prefill=meta_all['missing_frac_prefill'].to_numpy(dtype=np.float32),
)
print(f'Clinical feature matrix: {clin_all.shape}')

## 13. Classical ML Baseline Evaluator

In [ ]:
def make_group_aware_inner_splitter(n_splits: int = 3, random_state: int = GLOBAL_SEED):
    if HAS_STRATIFIED_GROUP_KFOLD:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    from sklearn.model_selection import GroupKFold
    return GroupKFold(n_splits=n_splits)

In [ ]:
def evaluate_sklearn_baseline(
    model_class,
    model_kwargs: dict,
    X_features: np.ndarray,
    y: np.ndarray,
    metadata: pd.DataFrame,
    folds: List[Tuple[np.ndarray, np.ndarray]],
    model_name: str = 'Baseline',
    scale_features: bool = True,
    param_distributions=None,
    n_search_iter: int = 30,
    inner_cv: int = 3,
) -> List[dict]:
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    fold_results = []
    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f'  {model_name} -- Fold {fold_idx + 1}/{len(folds)}')
        X_tr = X_features[train_idx]
        X_va = X_features[val_idx]
        y_tr = y[train_idx]
        y_va = y[val_idx]
        meta_tr  = meta.iloc[train_idx].reset_index(drop=True)
        meta_val = meta.iloc[val_idx].reset_index(drop=True)
        groups_tr = meta_tr['rec_id'].astype(str).to_numpy()
        if scale_features:
            scaler = StandardScaler()
            X_tr   = scaler.fit_transform(X_tr)
            X_va   = scaler.transform(X_va)
        if param_distributions is not None:
            inner_splitter = make_group_aware_inner_splitter(n_splits=inner_cv, random_state=GLOBAL_SEED + fold_idx)
            search = RandomizedSearchCV(
                estimator=model_class(**model_kwargs),
                param_distributions=param_distributions,
                n_iter=n_search_iter,
                cv=inner_splitter,
                scoring='f1_macro',
                random_state=GLOBAL_SEED,
                n_jobs=-1,
                refit=True,
            )
            search.fit(X_tr, y_tr, groups=groups_tr)
            clf = search.best_estimator_
            print(f'    Best params: {search.best_params_}  (inner F1={search.best_score_:.4f})')
        else:
            clf = model_class(**model_kwargs)
            clf.fit(X_tr, y_tr)
        y_proba_val = clf.predict_proba(X_va)
        _, step_y_true, step_y_pred = aggregate_to_record_step_level(y_proba_val, meta_val)
        metrics = compute_metrics(step_y_true, step_y_pred)
        metrics['fold'] = fold_idx + 1
        fold_results.append(metrics)
        print(
            f'    Acc={metrics["accuracy"]:.4f}  BalAcc={metrics["balanced_accuracy"]:.4f}  '
            f'F1={metrics["macro_f1"]:.4f}  '
            f'R_N={metrics["recall_Normal"]:.3f}  R_M={metrics["recall_Mild"]:.3f}  R_S={metrics["recall_Severe"]:.3f}'
        )
    return fold_results

## 14. Classical ML Baselines

In [ ]:
lr_param_dist = [
    {'C': loguniform(1e-3, 1e3), 'penalty': ['l1'], 'solver': ['saga']},
    {'C': loguniform(1e-3, 1e3), 'penalty': ['l2'], 'solver': ['lbfgs', 'saga']},
    {'C': loguniform(1e-3, 1e3), 'penalty': ['elasticnet'], 'solver': ['saga'], 'l1_ratio': uniform(0.1, 0.8)},
]

lr_results = evaluate_sklearn_baseline(
    model_class=LogisticRegression,
    model_kwargs=dict(multi_class='multinomial', max_iter=3000, class_weight='balanced', random_state=GLOBAL_SEED),
    X_features=clin_all,
    y=y_all,
    metadata=meta_all,
    folds=baseline_folds,
    model_name='Logistic Regression',
    scale_features=True,
    param_distributions=lr_param_dist,
    n_search_iter=40,
    inner_cv=3,
)
lr_summary = summarise_fold_results(lr_results, 'Logistic Regression')
print(f'\nLR -- Macro F1: {lr_summary["macro_f1_mean"]:.4f} +/- {lr_summary["macro_f1_std"]:.4f}')

In [ ]:
svm_param_dist = {
    'C': loguniform(1e-2, 1e3),
    'kernel': ['rbf', 'linear', 'poly'],
    'gamma': ['scale', 'auto', 1e-4, 1e-3, 1e-2, 1e-1, 1],
    'degree': [2, 3],
}

svm_results = evaluate_sklearn_baseline(
    model_class=SVC,
    model_kwargs=dict(probability=True, class_weight='balanced', random_state=GLOBAL_SEED),
    X_features=clin_all,
    y=y_all,
    metadata=meta_all,
    folds=baseline_folds,
    model_name='SVM',
    scale_features=True,
    param_distributions=svm_param_dist,
    n_search_iter=50,
    inner_cv=3,
)
svm_summary = summarise_fold_results(svm_results, 'SVM')
print(f'\nSVM -- Macro F1: {svm_summary["macro_f1_mean"]:.4f} +/- {svm_summary["macro_f1_std"]:.4f}')

In [ ]:
rf_param_dist = {
    'n_estimators': [100, 200, 300, 500, 800],
    'max_depth': [None, 10, 20, 30, 50, 70],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None, 0.5],
}

rf_results = evaluate_sklearn_baseline(
    model_class=RandomForestClassifier,
    model_kwargs=dict(class_weight='balanced_subsample', random_state=GLOBAL_SEED, n_jobs=-1),
    X_features=clin_all,
    y=y_all,
    metadata=meta_all,
    folds=baseline_folds,
    model_name='Random Forest',
    scale_features=False,
    param_distributions=rf_param_dist,
    n_search_iter=60,
    inner_cv=3,
)
rf_summary = summarise_fold_results(rf_results, 'Random Forest')
print(f'\nRF -- Macro F1: {rf_summary["macro_f1_mean"]:.4f} +/- {rf_summary["macro_f1_std"]:.4f}')

## 15. ResNet1D Baseline

In [ ]:
def evaluate_resnet_outer_folds(
    build_fn,
    X_signals: np.ndarray,
    y: np.ndarray,
    metadata: pd.DataFrame,
    folds: List[Tuple[np.ndarray, np.ndarray]],
    model_name: str = 'ResNet1D',
    epochs: int = 100,
    batch_size: int = 16,
    patience: int = 20,
    lr: float = 1e-3,
) -> List[dict]:
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    fold_results = []
    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f'  {model_name} -- Fold {fold_idx + 1}/{len(folds)}')
        X_tr  = X_signals[train_idx]
        X_va  = X_signals[val_idx]
        y_tr  = y[train_idx]
        y_va  = y[val_idx]
        meta_val = meta.iloc[val_idx].reset_index(drop=True)
        norm_stats = compute_norm_stats(X_tr)
        X_tr_n = apply_norm(X_tr, norm_stats)
        X_va_n = apply_norm(X_va, norm_stats)
        if CFG.get('augment_train', True):
            X_tr_n = augment_signal(X_tr_n, CFG)
        classes     = np.unique(y_tr)
        weights     = compute_class_weight('balanced', classes=classes, y=y_tr)
        class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
        y_tr_cat = to_categorical(y_tr, NUM_CLASSES)
        y_va_cat = to_categorical(y_va, NUM_CLASSES)
        tf.keras.backend.clear_session()
        model = build_fn()
        model.compile(
            optimizer=Adam(learning_rate=lr),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=CFG['label_smoothing']),
            metrics=['accuracy'],
        )
        rs_cb = RecordStepBalancedAccuracy(X_val=X_va_n, y_val=y_va, windows_per_step=CONTROLLED_WINDOWS_PER_STEP, batch_size=32, verbose=False)
        cbs = get_callbacks(patience=patience, monitor='val_record_step_balanced_accuracy', monitor_mode='max', prefix_callbacks=[rs_cb])
        model.fit(
            X_tr_n, y_tr_cat,
            validation_data=(X_va_n, y_va_cat),
            epochs=epochs,
            batch_size=batch_size,
            class_weight=class_weight,
            callbacks=cbs,
            verbose=0,
        )
        y_proba_val = model.predict(X_va_n, batch_size=32, verbose=0)
        _, step_y_true, step_y_pred = aggregate_to_record_step_level(y_proba_val, meta_val)
        metrics = compute_metrics(step_y_true, step_y_pred)
        metrics['fold'] = fold_idx + 1
        fold_results.append(metrics)
        print(
            f'    Acc={metrics["accuracy"]:.4f}  BalAcc={metrics["balanced_accuracy"]:.4f}  '
            f'F1={metrics["macro_f1"]:.4f}  '
            f'R_N={metrics["recall_Normal"]:.3f}  R_M={metrics["recall_Mild"]:.3f}  R_S={metrics["recall_Severe"]:.3f}'
        )
        del model
        tf.keras.backend.clear_session()
        gc.collect()
    return fold_results

In [ ]:
resnet_build_fn = lambda: build_resnet1d(dropout_rate=0.40, kernel_size=7, base_filters=64)

resnet_results = evaluate_resnet_outer_folds(
    build_fn=resnet_build_fn,
    X_signals=X_all,
    y=y_all,
    metadata=meta_all,
    folds=baseline_folds,
    model_name='ResNet1D',
    epochs=CFG['scratch_epochs'],
    batch_size=CFG['scratch_batch'],
    patience=CFG['scratch_patience'],
    lr=CFG['scratch_lr'],
)
resnet_summary = summarise_fold_results(resnet_results, 'ResNet1D')
print(f'\nResNet1D -- Macro F1: {resnet_summary["macro_f1_mean"]:.4f} +/- {resnet_summary["macro_f1_std"]:.4f}')

## 16. Final Model (Hybrid CNN with Clinical Features)

In [ ]:
def evaluate_final_model(
    X_signals: np.ndarray,
    y: np.ndarray,
    metadata: pd.DataFrame,
    clin_feats: np.ndarray,
    folds: List[Tuple[np.ndarray, np.ndarray]],
    model_name: str = 'Final Model',
    epochs: int = 100,
    batch_size: int = 16,
    patience: int = 20,
    lr: float = 1e-3,
) -> List[dict]:
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    fold_results = []
    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f'  {model_name} -- Fold {fold_idx + 1}/{len(folds)}')
        X_tr   = X_signals[train_idx]
        X_va   = X_signals[val_idx]
        y_tr   = y[train_idx]
        y_va   = y[val_idx]
        c_tr   = clin_feats[train_idx]
        c_va   = clin_feats[val_idx]
        meta_val = meta.iloc[val_idx].reset_index(drop=True)
        norm_stats = compute_norm_stats(X_tr)
        X_tr_n = apply_norm(X_tr, norm_stats)
        X_va_n = apply_norm(X_va, norm_stats)
        c_tr_n, _ = normalize_clinical_features(c_tr, c_tr)
        c_va_n, _ = normalize_clinical_features(c_tr, c_va)
        if CFG.get('augment_train', True):
            X_tr_n = augment_signal(X_tr_n, CFG)
        classes      = np.unique(y_tr)
        weights      = compute_class_weight('balanced', classes=classes, y=y_tr)
        class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
        y_tr_cat = to_categorical(y_tr, NUM_CLASSES)
        y_va_cat = to_categorical(y_va, NUM_CLASSES)
        tf.keras.backend.clear_session()
        model = build_model(
            input_length=CFG['segment_length'],
            num_classes=NUM_CLASSES,
            temporal_filters=CFG['temporal_filters'],
            temporal_kernel=CFG['temporal_kernel'],
            separable_filters=CFG['separable_filters'],
            separable_kernels=CFG['separable_kernels'],
            projection_filters=CFG['projection_filters'],
            dropout_rate=CFG['dropout_rate'],
            n_clinical_features=N_CLINICAL_FEATURES,
            residual_dilation_rates=CFG['residual_dilation_rates'],
        )
        model.compile(
            optimizer=Adam(learning_rate=lr),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=CFG['label_smoothing']),
            metrics=['accuracy'],
        )
        rs_cb = RecordStepBalancedAccuracy(
            X_val=[X_va_n, c_va_n], y_val=y_va, windows_per_step=CONTROLLED_WINDOWS_PER_STEP, batch_size=32, verbose=False
        )
        cbs = get_callbacks(patience=patience, monitor='val_record_step_balanced_accuracy', monitor_mode='max', prefix_callbacks=[rs_cb])
        model.fit(
            [X_tr_n, c_tr_n], y_tr_cat,
            validation_data=([X_va_n, c_va_n], y_va_cat),
            epochs=epochs,
            batch_size=batch_size,
            class_weight=class_weight,
            callbacks=cbs,
            verbose=0,
        )
        y_proba_val = model.predict([X_va_n, c_va_n], batch_size=32, verbose=0)
        _, step_y_true, step_y_pred = aggregate_to_record_step_level(y_proba_val, meta_val)
        metrics = compute_metrics(step_y_true, step_y_pred)
        metrics['fold'] = fold_idx + 1
        fold_results.append(metrics)
        print(
            f'    Acc={metrics["accuracy"]:.4f}  BalAcc={metrics["balanced_accuracy"]:.4f}  '
            f'F1={metrics["macro_f1"]:.4f}  '
            f'R_N={metrics["recall_Normal"]:.3f}  R_M={metrics["recall_Mild"]:.3f}  R_S={metrics["recall_Severe"]:.3f}'
        )
        del model
        tf.keras.backend.clear_session()
        gc.collect()
    return fold_results

In [ ]:
final_results = evaluate_final_model(
    X_signals=X_all,
    y=y_all,
    metadata=meta_all,
    clin_feats=clin_all,
    folds=baseline_folds,
    model_name='Final Model',
    epochs=CFG['scratch_epochs'],
    batch_size=CFG['scratch_batch'],
    patience=CFG['scratch_patience'],
    lr=CFG['scratch_lr'],
)
final_summary = summarise_fold_results(final_results, 'Final Model')
print(f'\nFinal Model -- Macro F1: {final_summary["macro_f1_mean"]:.4f} +/- {final_summary["macro_f1_std"]:.4f}')

## 17. Comparison Table and Plots

In [ ]:
all_summaries = [lr_summary, svm_summary, rf_summary, resnet_summary, final_summary]
comparison_df = pd.DataFrame(all_summaries).set_index('model')

display_rows = []
for model_name in comparison_df.index:
    row = comparison_df.loc[model_name]
    display_rows.append({
        'Model':         model_name,
        'Accuracy':      f"{row['accuracy_mean']:.4f} +/- {row['accuracy_std']:.4f}",
        'Bal. Accuracy': f"{row['balanced_accuracy_mean']:.4f} +/- {row['balanced_accuracy_std']:.4f}",
        'Macro F1':      f"{row['macro_f1_mean']:.4f} +/- {row['macro_f1_std']:.4f}",
        'Normal Recall': f"{row['normal_recall_mean']:.4f}",
        'Mild Recall':   f"{row['mild_recall_mean']:.4f}",
        'Severe Recall': f"{row['severe_recall_mean']:.4f}",
    })
display_df = pd.DataFrame(display_rows).set_index('Model')
print('\n' + '=' * 90)
print('MODEL COMPARISON -- Record-Step Level (mean +/- std across folds)')
print('=' * 90)
print(display_df.to_string())

comparison_df.to_csv(OUTPUT_DIR / 'model_comparison.csv')
display_df.to_csv(OUTPUT_DIR / 'model_comparison_formatted.csv')
print(f'\nSaved to {OUTPUT_DIR}')

In [ ]:
model_names = comparison_df.index.tolist()
x_pos = np.arange(len(model_names))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metric_info = [
    ('balanced_accuracy_mean', 'balanced_accuracy_std', 'Balanced Accuracy', 'darkorange'),
    ('macro_f1_mean',          'macro_f1_std',          'Macro F1',          'steelblue'),
    ('accuracy_mean',          'accuracy_std',          'Accuracy',          'seagreen'),
]
for ax, (mean_col, std_col, title, color) in zip(axes, metric_info):
    ax.bar(x_pos, comparison_df[mean_col], yerr=comparison_df[std_col], capsize=4,
           color=color, edgecolor='black', alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(title)
    ax.set_title(f'{title} (mean +/- std)')
    ax.grid(axis='y', alpha=0.3)
fig.suptitle('Model Comparison -- Record-Step Level', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'model_comparison_barplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig2, ax2 = plt.subplots(figsize=(13, 6))
bar_width   = 0.15
class_colors = {'Normal': '#2ca02c', 'Mild': '#ff7f0e', 'Severe': '#d62728'}
for i, cls_name in enumerate(CLASS_NAMES):
    col    = f'{cls_name.lower()}_recall_mean'
    offset = (i - 1) * bar_width
    ax2.bar(x_pos + offset, comparison_df[col], width=bar_width,
            label=f'{cls_name} Recall', color=class_colors[cls_name], edgecolor='black', alpha=0.85)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(model_names, rotation=30, ha='right', fontsize=10)
ax2.set_ylabel('Recall')
ax2.set_title('Per-Class Recall by Model')
ax2.legend(frameon=False)
ax2.grid(axis='y', alpha=0.3)
fig2.tight_layout()
fig2.savefig(OUTPUT_DIR / 'model_comparison_recall.png', dpi=300, bbox_inches='tight')
plt.show()
print('\nModel comparison complete.')

# CTU-UHB Step 2+3 Multiclass Short Experiment Plan


Minimal refactor of the CTU-UHB multiclass pipeline for a focused dissertation-ready experiment.


## Goal

Improve **step-level / record-step 3-class performance**, especially **Mild recall**, while keeping grouped cross-validation by `rec_id` and the existing preprocessing logic.


## Minimal changes in this version

- Drop **Step 1** entirely

- Train and evaluate on **Step 2 + Step 3** 30-minute windows only, using **3 controlled windows per step**

- Increase the window-quality threshold to **`max_nan_fraction = 0.16`**

- Keep the standard **3-class softmax** CNN as the primary model

- Add a compact **engineered-feature branch** alongside the raw-signal CNN branch

- Standardize engineered features using **training-fold statistics only**

- Keep **grouped CV by `rec_id`** but evaluate at the **record-step level** rather than collapsing back to one label per record


## 1. Imports

In [1]:
# ── Disable oneDNN/MKL ops BEFORE importing TF ──────────────────
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

# Core
import gc
import math
import numpy as np
import pandas as pd
from pathlib import Path
import ast
import wfdb
import json
import warnings
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

# Signal processing
from scipy.signal import find_peaks
from scipy.interpolate import interp1d

# Sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, roc_auc_score, roc_curve, auc,
)
from sklearn.utils.class_weight import compute_class_weight

# Try to import StratifiedGroupKFold (sklearn >= 1.0)
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
    print("StratifiedGroupKFold available - will use group-aware stratified splitting.")
except ImportError:
    from sklearn.model_selection import GroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = False
    print("WARNING: StratifiedGroupKFold not available (sklearn < 1.0).")
    print("         Falling back to GroupKFold - class balance across folds not guaranteed.")

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.layers import (
    Conv1D, SeparableConv1D, BatchNormalization, Activation,
    AveragePooling1D, Dropout, GlobalAveragePooling1D, Dense,
    Concatenate, Reshape, Multiply, Add,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, Callback,
)
from tensorflow.keras.utils import to_categorical

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)
warnings.filterwarnings('ignore')

print(f"TensorFlow {tf.__version__} | NumPy {np.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

StratifiedGroupKFold available - will use group-aware stratified splitting.


c:\Users\ZEYNEP\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


TensorFlow 2.16.1 | NumPy 1.26.4
GPU available: False


## 2. Configuration

All experiment parameters in one place. Change values here to toggle between
from-scratch training and transfer learning, adjust segmentation, etc.

In [2]:
# ═══════════════════════════════════════════════════════════════════
# PATHS
# ═══════════════════════════════════════════════════════════════════

RAW_DATASET   = Path("../data/raw_dataset")
STEP2_LABELS  = Path("../ExpertAnnotations/step2_labels.csv")
STEP3_LABELS  = Path("../ExpertAnnotations/step3_labels.csv")
PTBXL_PATH    = Path("../ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")  # legacy / optional
OUTPUT_DIR    = Path("outputs/step23_softmax_short_plan")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════
# SIGNAL PARAMETERS
# ═══════════════════════════════════════════════════════════════════

RAW_SAMPLING_RATE      = 4
TARGET_SAMPLING_RATE   = 1
DOWNSAMPLE_FACTOR      = RAW_SAMPLING_RATE // TARGET_SAMPLING_RATE

WINDOW_MINUTES  = 30
WINDOW_LENGTH   = WINDOW_MINUTES * 60 * TARGET_SAMPLING_RATE

# Keep the existing key names to minimize downstream code changes.
SAMPLING_RATE   = TARGET_SAMPLING_RATE
SEGMENT_MINUTES = WINDOW_MINUTES
SEGMENT_LENGTH  = WINDOW_LENGTH

# ═══════════════════════════════════════════════════════════════════
# LABEL DEFINITIONS
# ═══════════════════════════════════════════════════════════════════

NUM_CLASSES = 3
CLASS_NAMES = ["Normal", "Mild", "Severe"]
LABEL_MAP   = {1: 0, 2: 1, 3: 2}

# Legacy PTB-XL mapping retained for compatibility with older helper cells.
PTBXL_CLASS_NAMES = ["Normal", "Intermediate", "Pathological"]

# ═══════════════════════════════════════════════════════════════════
# EXPERIMENT CONFIGURATION
# ═══════════════════════════════════════════════════════════════════

CFG = dict(
    # ── Step-window experiment scope ──────────────────────────────
    step_numbers           = [2, 3],
    mode_a_offsets         = {2: (60, 30), 3: (30, 0)},
    controlled_step_offsets = {
        2: [(70, 40), (60, 30), (50, 20)],
        3: [(50, 20), (40, 10), (30, 0)],
    },
    segment_minutes        = SEGMENT_MINUTES,
    segment_length         = SEGMENT_LENGTH,
    max_nan_fraction       = 0.16,
    pad_short_records      = False,

    # ── Cross-Validation ──────────────────────────────────────────
    n_splits               = 10 ,
    cv_seed                = 42,
    min_severe_records     = 10,
    max_cv_attempts        = 100,

    # ── Model Architecture ────────────────────────────────────────
    temporal_filters       = 32,
    temporal_kernel        = 7,
    separable_filters      = 32,
    separable_kernels      = [3, 7, 15],
    projection_filters     = 32,
    dropout_rate           = 0.40,

    # ── Engineered-feature branch ────────────────────────────────
    use_clinical_features  = True,

    # ── Legacy transfer settings kept but disabled ───────────────
    use_adaptation_layers  = False,
    adaptation_units       = [64, 32],
    adaptation_dropout     = 0.30,
    skip_ptbxl_pretrain    = True,

    # ── Record-level reweighting / calibration ───────────────────
    oversample_severe      = False,
    severe_boost           = 0.0,

    # ── Augmentation Pipeline ────────────────────────────────────
    augment_train          = True,
    aug_time_warp          = True,
    aug_warp_sigma         = 0.10,
    aug_amplitude_scale    = True,
    aug_amp_sigma          = 0.10,
    aug_noise              = True,
    aug_noise_std          = 0.02,
    aug_signal_loss        = True,
    aug_gap_frac           = 0.05,
    aug_n_gaps             = 2,

    # ── Training: softmax baseline first ────────────────────────
    scratch_epochs         = 100,
    scratch_lr             = 1e-3,
    scratch_batch          = 2,
    scratch_patience       = 20,
    label_smoothing        = 0.05,
    use_class_weights      = True,

    # ── Legacy transfer hyperparameters retained for compatibility ─
    s0_epochs              = 15,
    s0_lr                  = 1e-3,
    s0_batch               = 64,
    s0_patience            = 5,
    s1a_epochs             = 30,
    s1a_lr                 = 1e-4,
    s1a_batch              = 16,
    s1a_patience           = 10,
    s1b_epochs             = 40,
    s1b_lr                 = 5e-5,
    s1b_batch              = 8,
    s1b_patience           = 15,
    s1b_unfreeze_blocks    = 3,
    s1c_epochs             = 50,
    s1c_lr                 = 1e-5,
    s1c_batch              = 8,
    s1c_patience           = 20,
    warmup_epochs          = 5,
    warmup_start_lr        = 1e-6,
    s1a_augment            = False,
    s1b_augment            = True,
    s1b_noise_std          = 0.01,
    s1c_augment            = True,
    s1c_noise_std          = 0.02,
)

windows_per_step = {step_num: len(offsets) for step_num, offsets in CFG['controlled_step_offsets'].items()}
if len(set(windows_per_step.values())) != 1:
    raise ValueError(f"All steps must use the same number of controlled windows, got: {windows_per_step}")

CONTROLLED_WINDOWS_PER_STEP = next(iter(windows_per_step.values()))

print("═" * 60)
print("CONFIGURATION SUMMARY")
print("═" * 60)
print(f"  Windows used:      Steps {CFG['step_numbers']}")
print(f"  Sampling rate:     {RAW_SAMPLING_RATE} Hz raw -> {SAMPLING_RATE} Hz model input")
print(f"  Window length:     {CFG['segment_minutes']} min = {CFG['segment_length']} samples @ {SAMPLING_RATE} Hz")
print(f"  Canonical offsets: {CFG['mode_a_offsets']}")
print(f"  Controlled windows:{CONTROLLED_WINDOWS_PER_STEP} per step")
for step_num in CFG['step_numbers']:
    print(f"    Step {step_num}: {CFG['controlled_step_offsets'][step_num]}")
print(f"  Max NaN frac:      {CFG['max_nan_fraction']:.0%}")
print(f"  CV folds:          {CFG['n_splits']}")
print(f"  Min Severe/fold:   {CFG['min_severe_records']} records")
print(f"  Label smoothing:   {CFG['label_smoothing']:.2f}")
print(f"  Softmax baseline:  enabled")
print(f"  Feature branch:    compact engineered FHR features")
print(f"  Grouped CV:        by rec_id (unchanged)")
print(f"  PTB-XL legacy:     disabled in this short plan")
print("═" * 60)

════════════════════════════════════════════════════════════
CONFIGURATION SUMMARY
════════════════════════════════════════════════════════════
  Windows used:      Steps [2, 3]
  Sampling rate:     4 Hz raw -> 1 Hz model input
  Window length:     30 min = 1800 samples @ 1 Hz
  Canonical offsets: {2: (60, 30), 3: (30, 0)}
  Controlled windows:3 per step
    Step 2: [(70, 40), (60, 30), (50, 20)]
    Step 3: [(50, 20), (40, 10), (30, 0)]
  Max NaN frac:      16%
  CV folds:          10
  Min Severe/fold:   10 records
  Label smoothing:   0.05
  Softmax baseline:  enabled
  Feature branch:    compact engineered FHR features
  Grouped CV:        by rec_id (unchanged)
  PTB-XL legacy:     disabled in this short plan
════════════════════════════════════════════════════════════


## 4. Step-Window Extraction (Steps 2 + 3 Only)



**Strategy:**

- Extract only the **Step 2** and **Step 3** 30-minute delivery-anchored windows

- For each eligible record-step, create a fixed set of **3 controlled windows**

- Keep **grouping by `rec_id`** unchanged so all windows from one record stay in the same fold

- Reject the entire record-step if any controlled window is too short or exceeds the missing-data threshold

- Track `rec_id` and window metadata so validation predictions are still aggregated at the **record level**


In [3]:
# ═══════════════════════════════════════════════════════════════════
# DATA LOADING & FHR CLEANING
# ═══════════════════════════════════════════════════════════════════


def load_step_labels(step_paths: Dict[int, Path]) -> Dict[int, pd.DataFrame]:
    """Load Step 2 and Step 3 labels independently and filter uninterpretable rows."""
    step_labels = {}
    for step_num, path in step_paths.items():
        df = pd.read_csv(path)
        total = len(df)
        df = df[df['Majority_Vote_Label'] != -1].copy()
        df['class_id'] = df['Majority_Vote_Label'].map(LABEL_MAP)
        df['rec_id'] = df['rec_id'].astype(str)
        excluded = total - len(df)
        print(f"Step {step_num}: {len(df)} labelled records ({excluded} uninterpretable excluded)")
        for cls_id, name in enumerate(CLASS_NAMES):
            n = int((df['class_id'] == cls_id).sum())
            print(f"  Class {cls_id} ({name}): {n} ({n / max(len(df), 1) * 100:.1f}%)")
        step_labels[step_num] = df
    return step_labels


def remove_trailing_zeros(signal) -> list:
    """Remove trailing zeros (marks end of recording / delivery)."""
    sig = list(signal) if isinstance(signal, np.ndarray) else list(signal)
    i = len(sig) - 1
    while i >= 0 and sig[i] == 0:
        i -= 1
    return sig[:i + 1]


def clean_fhr(fhr_array, fs: int = 4) -> np.ndarray:
    """
    Clean a fetal heart rate signal using the existing pipeline.

    This refactor deliberately keeps preprocessing unchanged.
    """
    fhr = pd.Series(np.array(fhr_array, dtype=float))

    # Step 1: Replace 0 with NaN
    fhr.replace(0, np.nan, inplace=True)

    # Step 2: Remove NaN gaps > 15 consecutive seconds
    na = fhr.isnull()
    gap_groups = na.ne(na.shift()).cumsum()
    gap_sizes = fhr.groupby(gap_groups.values).transform('size')
    fhr = fhr[~(gap_sizes.ge(fs * 15 + 1) & na)].reset_index(drop=True)

    # Step 3: Outlier detection
    fhr[fhr < 50] = np.nan
    fhr[fhr > 200] = np.nan

    # Step 4: Linear interpolation
    fhr = fhr.interpolate(method='linear')

    # Step 5: Spike detection and removal
    diff = fhr - fhr.shift()
    fhr[(diff > 25) | (diff < -25)] = np.nan
    fhr = fhr.interpolate(method='linear')

    # Edge NaN fill
    fhr = fhr.ffill().bfill()

    return fhr.values


def downsample_signal(signal: np.ndarray, factor: int = 4) -> np.ndarray:
    """Downsample a cleaned 1D signal by simple decimation."""
    if factor <= 1:
        return np.asarray(signal, dtype=np.float32)
    return np.asarray(signal[::factor], dtype=np.float32)


def load_raw_signals(dataset_path: Path) -> Dict[str, dict]:
    """
    Load all CTU-UHB records, clean FHR at native 4 Hz, then downsample to 1 Hz.

    The returned structure now preserves raw and cleaned views so poster figures
    can show the preprocessing effect directly.

    Returns
    -------
    dict[rec_id] with raw 4 Hz, raw 1 Hz, cleaned 4 Hz, and cleaned 1 Hz signals.
    """
    records = [p.stem for p in dataset_path.glob("*.hea")]
    print(f"Found {len(records)} .hea files in {dataset_path}")

    data = {}
    for rid in sorted(records):
        try:
            rec = wfdb.rdrecord(str(dataset_path / rid))
            fhr_raw_4hz = np.asarray(remove_trailing_zeros(rec.p_signal[:, 0].tolist()), dtype=np.float32)
            if fhr_raw_4hz.size == 0:
                raise ValueError("empty FHR after trimming trailing zeros")

            fhr_raw_1hz = downsample_signal(fhr_raw_4hz, factor=DOWNSAMPLE_FACTOR)
            fhr_clean_4hz = np.asarray(clean_fhr(fhr_raw_4hz, fs=RAW_SAMPLING_RATE), dtype=np.float32)
            fhr_clean_1hz = downsample_signal(fhr_clean_4hz, factor=DOWNSAMPLE_FACTOR)

            data[rid] = {
                'FHR': fhr_clean_1hz,
                'FHR_1hz': fhr_clean_1hz,
                'FHR_raw_1hz': fhr_raw_1hz,
                'FHR_clean_4hz': fhr_clean_4hz,
                'FHR_raw_4hz': fhr_raw_4hz,
                'length': len(fhr_clean_1hz),
                'length_raw_1hz': len(fhr_raw_1hz),
                'length_4hz': len(fhr_clean_4hz),
            }
        except Exception as e:
            print(f"  Skipping {rid}: {e}")

    print(f"\nLoaded {len(data)} FHR signals")
    lengths = [d['length'] for d in data.values()]
    print(f"  Duration range: {min(lengths)/(SAMPLING_RATE*60):.1f} - {max(lengths)/(SAMPLING_RATE*60):.1f} minutes")
    print("  Stored views: raw 4 Hz, raw 1 Hz, cleaned 4 Hz, cleaned 1 Hz")
    return data


# -- Load data --
STEP_LABEL_PATHS = {2: STEP2_LABELS, 3: STEP3_LABELS}
step_label_dfs = load_step_labels(STEP_LABEL_PATHS)
signal_data = load_raw_signals(RAW_DATASET)

labelled_rec_ids = set()
for df in step_label_dfs.values():
    labelled_rec_ids.update(df['rec_id'].tolist())

valid_rec_ids = sorted(labelled_rec_ids & set(signal_data.keys()))
print(f"\nRecords with at least one Step 2/3 label and a signal: {len(valid_rec_ids)}")

Step 2: 548 labelled records (4 uninterpretable excluded)
  Class 0 (Normal): 229 (41.8%)
  Class 1 (Mild): 251 (45.8%)
  Class 2 (Severe): 68 (12.4%)
Step 3: 337 labelled records (215 uninterpretable excluded)
  Class 0 (Normal): 127 (37.7%)
  Class 1 (Mild): 153 (45.4%)
  Class 2 (Severe): 57 (16.9%)
Found 552 .hea files in ..\data\raw_dataset

Loaded 552 FHR signals
  Duration range: 33.8 - 90.1 minutes
  Stored views: raw 4 Hz, raw 1 Hz, cleaned 4 Hz, cleaned 1 Hz

Records with at least one Step 2/3 label and a signal: 549


## 4. Step-Window Construction at 1 Hz

**Strategy:**
- Clean raw FHR at its native **4 Hz**, then downsample to **1 Hz** for modeling.
- Extract a fixed set of **3 controlled 30-minute windows per eligible step** rather than a single canonical window.
- Step 2 uses earlier / canonical / later offsets around the canonical interval; Step 3 is delivery-bounded, so its three windows run from earlier coverage up to the canonical delivery window.
- If any of the 3 controlled windows for a record-step is too short or fails the missing-data threshold, that record-step is dropped entirely.
- The `group` array still tracks the parent `rec_id` for every window, so grouped CV remains leakage-free and validation windows are still aggregated back to the record level.

In [4]:
# ═══════════════════════════════════════════════════════════════════

# STEP-WINDOW EXTRACTION (STEPS 2 + 3 ONLY)

# ═══════════════════════════════════════════════════════════════════



def _window_bounds_from_end(

    sig_len: int,

    offsets_from_end_min: Tuple[int, int],

    fs: int = 4,

) -> Optional[Tuple[int, int]]:

    """Return sample bounds for one delivery-anchored window."""

    start_from_end_min, end_from_end_min = offsets_from_end_min

    if start_from_end_min <= end_from_end_min:

        raise ValueError(

            f"Expected start offset > end offset, got {offsets_from_end_min}"

        )



    start_from_end = start_from_end_min * 60 * fs

    end_from_end = end_from_end_min * 60 * fs



    if sig_len < start_from_end:

        return None



    start_idx = sig_len - start_from_end

    end_idx = sig_len - end_from_end if end_from_end > 0 else sig_len

    if end_idx <= start_idx:

        return None



    return int(start_idx), int(end_idx)





def _get_controlled_window_bounds(

    sig_len: int,

    step_num: int,

    controlled_step_offsets: Dict[int, List[Tuple[int, int]]],

    fs: int = 4,

) -> Optional[List[Tuple[int, Tuple[int, int], int, int]]]:

    """Return all controlled window bounds for one record-step."""

    if step_num not in controlled_step_offsets:

        raise KeyError(f"Missing controlled offsets for step {step_num}")



    bounds = []

    for window_index, offsets in enumerate(controlled_step_offsets[step_num]):

        cur_bounds = _window_bounds_from_end(sig_len, offsets, fs=fs)

        if cur_bounds is None:

            return None

        start_idx, end_idx = cur_bounds

        bounds.append((window_index, offsets, start_idx, end_idx))



    return bounds





def create_step_windows(

    signal_data: Dict[str, dict],

    step_label_dfs: Dict[int, pd.DataFrame],

    record_ids: List[str],

    window_length: int = 7200,

    step_numbers: Optional[List[int]] = None,

    step_offsets: Optional[Dict[int, Tuple[int, int]]] = None,

    controlled_step_offsets: Optional[Dict[int, List[Tuple[int, int]]]] = None,

    max_nan_fraction: float = 0.16,

    pad_short_records: bool = False,

    fs: int = 4,

) -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:

    """

    Extract controlled Step 2 and Step 3 30-minute windows from cleaned CTU-UHB FHR signals.



    Each eligible record-step contributes the same fixed number of windows. If any controlled

    window for a record-step is too short or exceeds the missing-data threshold, that entire

    record-step is dropped so records are not weighted inconsistently.

    """

    if step_numbers is None:

        step_numbers = [2, 3]

    if step_offsets is None:

        step_offsets = {2: (60, 30), 3: (30, 0)}

    if controlled_step_offsets is None:

        controlled_step_offsets = {

            step_num: [step_offsets[step_num]]

            for step_num in step_numbers

        }



    windows_per_step = {

        step_num: len(controlled_step_offsets[step_num])

        for step_num in step_numbers

    }

    if len(set(windows_per_step.values())) != 1:

        raise ValueError(

            f"All steps must use the same number of controlled windows, got: {windows_per_step}"

        )



    fixed_windows_per_step = next(iter(windows_per_step.values()))



    X_list = []

    y_list = []

    group_list = []

    meta_list = []



    accepted_record_steps = 0

    skipped_step_short = 0

    skipped_step_nan = 0



    for rid in record_ids:

        if rid not in signal_data:

            continue



        fhr = np.array(signal_data[rid]['FHR'], dtype=np.float64)

        sig_len = len(fhr)



        for step_num in step_numbers:

            step_df = step_label_dfs.get(step_num)

            if step_df is None:

                continue



            row = step_df[step_df['rec_id'] == rid]

            if row.empty:

                continue



            controlled_bounds = _get_controlled_window_bounds(

                sig_len,

                step_num,

                controlled_step_offsets,

                fs=fs,

            )

            if controlled_bounds is None:

                skipped_step_short += 1

                continue



            label = int(row['class_id'].iloc[0])

            step_windows = []

            step_failed_short = False

            step_failed_nan = False



            for window_index, offsets, start_idx, end_idx in controlled_bounds:

                window = fhr[start_idx:end_idx]



                if len(window) < window_length:

                    if pad_short_records:

                        pad_len = window_length - len(window)

                        window = np.pad(window, (pad_len, 0), mode='edge')

                    else:

                        step_failed_short = True

                        break



                missing_frac_prefill = float(np.mean(np.isnan(window) | (window == 0)))

                if missing_frac_prefill > max_nan_fraction:

                    step_failed_nan = True

                    break



                if np.any(np.isnan(window)):

                    s = pd.Series(window)

                    window = s.interpolate(method='linear').ffill().bfill().values



                step_windows.append({

                    'window': window.astype(np.float32),

                    'window_index': window_index,

                    'offset_start_min': int(offsets[0]),

                    'offset_end_min': int(offsets[1]),

                    'window_start': start_idx,

                    'window_end': end_idx,

                    'missing_frac_prefill': missing_frac_prefill,

                })



            if step_failed_short:

                skipped_step_short += 1

                continue



            if step_failed_nan:

                skipped_step_nan += 1

                continue



            if len(step_windows) != fixed_windows_per_step:

                raise RuntimeError(

                    f"Expected {fixed_windows_per_step} windows for {rid} step {step_num}, "

                    f"got {len(step_windows)}"

                )



            accepted_record_steps += 1

            for step_window in step_windows:

                X_list.append(step_window['window'])

                y_list.append(label)

                group_list.append(rid)

                meta_list.append({

                    'rec_id': rid,

                    'step': step_num,

                    'window_index': step_window['window_index'],

                    'offset_start_min': step_window['offset_start_min'],

                    'offset_end_min': step_window['offset_end_min'],

                    'window_start': step_window['window_start'],

                    'window_end': step_window['window_end'],

                    'label': label,

                    'missing_frac_prefill': step_window['missing_frac_prefill'],

                })



    if X_list:

        X_windows = np.array(X_list, dtype=np.float32)[:, :, np.newaxis]

    else:

        X_windows = np.empty((0, window_length, 1), dtype=np.float32)

    y_windows = np.array(y_list, dtype=np.int32)

    groups = np.array(group_list)

    metadata = pd.DataFrame(meta_list)



    print("\nStep-window extraction complete:")

    print(f"  Total windows:             {len(X_windows)} from {len(set(group_list))} records")

    print(f"  Window shape:              {X_windows.shape}")

    print(f"  Controlled windows/step:   {fixed_windows_per_step}")

    print(f"  Accepted record-steps:     {accepted_record_steps}")

    print(f"  Skipped record-steps (short): {skipped_step_short}")

    print(f"  Skipped record-steps (NaN):   {skipped_step_nan}")

    if len(metadata) > 0:

        for step_num in step_numbers:

            step_mask = metadata['step'] == step_num

            n_step_windows = int(step_mask.sum())

            n_step_records = int(metadata.loc[step_mask, 'rec_id'].nunique())

            print(f"  Step {step_num}:               {n_step_windows} windows from {n_step_records} records")

        for cls_id, name in enumerate(CLASS_NAMES):

            n_class = int(np.sum(y_windows == cls_id))

            print(f"  Class {cls_id} ({name}):      {n_class} windows")



    return X_windows, y_windows, groups, metadata





# ── Create Step 2 + Step 3 windows from all valid records ──

print("Creating Step 2 + Step 3 controlled 30-minute windows...")

X_all, y_all, groups_all, meta_all = create_step_windows(

    signal_data=signal_data,

    step_label_dfs=step_label_dfs,

    record_ids=valid_rec_ids,

    window_length=CFG['segment_length'],

    step_numbers=CFG['step_numbers'],

    step_offsets=CFG['mode_a_offsets'],

    controlled_step_offsets=CFG['controlled_step_offsets'],

    max_nan_fraction=CFG['max_nan_fraction'],

    pad_short_records=CFG['pad_short_records'],

    fs=SAMPLING_RATE,

)



windows_per_record = pd.Series(groups_all).value_counts()

if len(windows_per_record) > 0:

    print(f"\nWindows per record: min={windows_per_record.min()}, max={windows_per_record.max()}, mean={windows_per_record.mean():.1f}")

    print("Step counts:")

    print(meta_all['step'].value_counts().sort_index())

    print("Window index counts:")

    print(meta_all['window_index'].value_counts().sort_index())

Creating Step 2 + Step 3 controlled 30-minute windows...

Step-window extraction complete:
  Total windows:             1398 from 336 records
  Window shape:              (1398, 1800, 1)
  Controlled windows/step:   3
  Accepted record-steps:     466
  Skipped record-steps (short): 419
  Skipped record-steps (NaN):   0
  Step 2:               399 windows from 133 records
  Step 3:               999 windows from 333 records
  Class 0 (Normal):      564 windows
  Class 1 (Mild):      636 windows
  Class 2 (Severe):      198 windows

Windows per record: min=3, max=6, mean=4.2
Step counts:
step
2    399
3    999
Name: count, dtype: int64
Window index counts:
window_index
0    466
1    466
2    466
Name: count, dtype: int64


## 6. Normalization, Augmentation & Engineered Features



**Normalization:** Z-score statistics are computed from the **training fold only**.



**Time-Series Augmentations** remain unchanged and are applied to training windows only.



**Compact engineered features** are extracted from the cleaned 30-minute FHR window:

1. mean

2. median

3. std

4. IQR

5. mean absolute successive difference

6. std of 1-minute means

7. number of accelerations

8. number of decelerations

9. max deceleration depth

10. fraction of missing/invalid samples before final fill


In [5]:
# ═══════════════════════════════════════════════════════════════════

# LEAKAGE-SAFE NORMALIZATION + AUGMENTATION + ENGINEERED FEATURES

# ═══════════════════════════════════════════════════════════════════



def compute_norm_stats(X_train: np.ndarray) -> dict:

    """Compute mean and std from TRAINING windows only."""

    vals = X_train.ravel()

    mean = float(np.mean(vals))

    std = float(np.std(vals))

    if std < 1e-8:

        std = 1.0

    return {'mean': mean, 'std': std}





def apply_norm(X: np.ndarray, stats: dict) -> np.ndarray:

    """Apply pre-computed z-score normalization."""

    X_norm = X.astype(np.float32)

    X_norm -= stats['mean']

    X_norm /= stats['std']

    return X_norm





# ═══════════════════════════════════════════════════════════════════

# TIME-SERIES AUGMENTATIONS (applied to training data only)

# ═══════════════════════════════════════════════════════════════════



def time_warp(X: np.ndarray, sigma: float = 0.1) -> np.ndarray:

    """Time warping: stretch/compress the time axis by smooth random amounts."""

    N, T, C = X.shape

    X_warped = np.empty_like(X)

    orig_steps = np.arange(T)

    n_knots = 4

    knot_pos = np.linspace(0, T - 1, n_knots + 2)



    for i in range(N):

        warp_factors = np.random.normal(loc=1.0, scale=sigma, size=n_knots + 2)

        warp_factors[0] = 1.0

        warp_factors[-1] = 1.0

        warped_knots = np.cumsum(np.diff(knot_pos) * warp_factors[:-1])

        warped_knots = np.concatenate([[0], warped_knots])

        warped_knots = warped_knots / warped_knots[-1] * (T - 1)

        interp_fn = interp1d(warped_knots, knot_pos, kind='linear', fill_value='extrapolate')

        warped_steps = np.clip(interp_fn(orig_steps), 0, T - 1)



        for c in range(C):

            interp_sig = interp1d(orig_steps, X[i, :, c], kind='linear', fill_value='extrapolate')

            X_warped[i, :, c] = interp_sig(warped_steps)



    return X_warped.astype(np.float32)





def amplitude_scale(X: np.ndarray, sigma: float = 0.1) -> np.ndarray:

    """Random per-sample amplitude scaling."""

    N = X.shape[0]

    scales = np.random.normal(1.0, sigma, size=(N, 1, 1)).astype(np.float32)

    return X * scales





def additive_noise(X: np.ndarray, noise_std: float = 0.02) -> np.ndarray:

    """Add mild Gaussian noise."""

    noise = np.random.normal(0, noise_std, X.shape).astype(np.float32)

    return X + noise





def signal_loss_simulation(

    X: np.ndarray,

    max_gap_frac: float = 0.05,

    n_gaps: int = 2,

) -> np.ndarray:

    """Simulate random signal dropout with interpolation across short gaps."""

    N, T, C = X.shape

    X_aug = X.copy()

    max_gap = max(1, int(T * max_gap_frac))



    for i in range(N):

        for _ in range(n_gaps):

            gap_len = np.random.randint(1, max_gap + 1)

            start = np.random.randint(0, T - gap_len)

            end = start + gap_len

            for c in range(C):

                left_val = X_aug[i, max(start - 1, 0), c]

                right_val = X_aug[i, min(end, T - 1), c]

                X_aug[i, start:end, c] = np.linspace(left_val, right_val, gap_len)



    return X_aug.astype(np.float32)





def augment_signal(X: np.ndarray, cfg: dict) -> np.ndarray:

    """Apply the unchanged augmentation pipeline to training data only."""

    X_aug = X.copy()

    if cfg.get('aug_time_warp', True):

        X_aug = time_warp(X_aug, sigma=cfg.get('aug_warp_sigma', 0.1))

    if cfg.get('aug_amplitude_scale', True):

        X_aug = amplitude_scale(X_aug, sigma=cfg.get('aug_amp_sigma', 0.1))

    if cfg.get('aug_noise', True):

        X_aug = additive_noise(X_aug, noise_std=cfg.get('aug_noise_std', 0.02))

    if cfg.get('aug_signal_loss', True):

        X_aug = signal_loss_simulation(

            X_aug,

            max_gap_frac=cfg.get('aug_gap_frac', 0.05),

            n_gaps=cfg.get('aug_n_gaps', 2),

        )

    return X_aug





# ═══════════════════════════════════════════════════════════════════

# COMPACT ENGINEERED FEATURE EXTRACTION

# ═══════════════════════════════════════════════════════════════════



def compute_clinical_features(

    X: np.ndarray,

    fs: int = 4,

    missing_frac_prefill: Optional[np.ndarray] = None,

) -> np.ndarray:

    """

    Extract a compact engineered feature vector from each cleaned FHR window.



    Features:

      1. mean

      2. median

      3. std

      4. IQR

      5. mean absolute successive difference

      6. std of 1-minute means

      7. number of accelerations

      8. number of decelerations

      9. max deceleration depth

     10. fraction of missing/invalid samples before final fill

    """

    N, T, _ = X.shape

    features = np.zeros((N, 10), dtype=np.float32)

    min_dur_samples = 15 * fs

    one_min_samples = 60 * fs



    if missing_frac_prefill is None:

        missing_frac_prefill = np.zeros(N, dtype=np.float32)

    else:

        missing_frac_prefill = np.asarray(missing_frac_prefill, dtype=np.float32)



    for i in range(N):

        sig = np.asarray(X[i, :, 0], dtype=np.float64)

        baseline = float(np.median(sig))

        deviation = sig - baseline



        features[i, 0] = float(np.mean(sig))

        features[i, 1] = float(np.median(sig))

        features[i, 2] = float(np.std(sig))

        q25, q75 = np.percentile(sig, [25, 75])

        features[i, 3] = float(q75 - q25)

        features[i, 4] = float(np.mean(np.abs(np.diff(sig)))) if len(sig) > 1 else 0.0



        n_minutes = T // one_min_samples

        if n_minutes >= 2:

            minute_means = [

                np.mean(sig[j * one_min_samples:(j + 1) * one_min_samples])

                for j in range(n_minutes)

            ]

            features[i, 5] = float(np.std(minute_means))

        else:

            features[i, 5] = features[i, 2]



        accel_mask = deviation >= 15

        decel_mask = deviation <= -15

        accel_count, _ = _count_episodes(deviation, accel_mask, min_dur_samples)

        decel_count, _ = _count_episodes(-deviation, decel_mask, min_dur_samples)

        features[i, 6] = float(accel_count)

        features[i, 7] = float(decel_count)

        features[i, 8] = float(np.max(np.maximum(-deviation, 0.0)))

        features[i, 9] = float(missing_frac_prefill[i])



    return features





def _count_episodes(

    deviation: np.ndarray,

    mask: np.ndarray,

    min_dur: int,

) -> tuple:

    """Count episodes where mask is True for at least min_dur consecutive samples."""

    count = 0

    total_area = 0.0

    in_episode = False

    episode_start = 0



    for j in range(len(mask)):

        if mask[j] and not in_episode:

            in_episode = True

            episode_start = j

        elif not mask[j] and in_episode:

            in_episode = False

            dur = j - episode_start

            if dur >= min_dur:

                count += 1

                total_area += float(np.sum(np.abs(deviation[episode_start:j])))



    if in_episode:

        dur = len(mask) - episode_start

        if dur >= min_dur:

            count += 1

            total_area += float(np.sum(np.abs(deviation[episode_start:])))



    return count, total_area





CLINICAL_FEATURE_NAMES = [

    'mean_fhr',

    'median_fhr',

    'std_fhr',

    'iqr_fhr',

    'masd',

    'minute_mean_std',

    'n_accel',

    'n_decel',

    'max_decel_depth',

    'missing_frac_prefill',

]

N_CLINICAL_FEATURES = len(CLINICAL_FEATURE_NAMES)





def normalize_clinical_features(

    feats_train: np.ndarray,

    feats_apply: np.ndarray,

) -> tuple:

    """Z-score normalize engineered features using TRAIN statistics only."""

    mean = feats_train.mean(axis=0)

    std = feats_train.std(axis=0)

    std[std < 1e-8] = 1.0

    return (feats_apply - mean) / std, {'mean': mean, 'std': std}





print("Normalization, augmentation & engineered-feature utilities defined:")

print(f"  Engineered features ({N_CLINICAL_FEATURES}): {CLINICAL_FEATURE_NAMES}")

print("  augment_signal() remains unchanged and is applied to training windows only")


Normalization, augmentation & engineered-feature utilities defined:
  Engineered features (10): ['mean_fhr', 'median_fhr', 'std_fhr', 'iqr_fhr', 'masd', 'minute_mean_std', 'n_accel', 'n_decel', 'max_decel_depth', 'missing_frac_prefill']
  augment_signal() remains unchanged and is applied to training windows only


## 9. Evaluation & Metrics

**Key principle:** Evaluate at the **record-step level**, not by forcing one label per full record.

During inference:
1. Predict window-level probabilities
2. Group windows by parent **record-step** (`rec_id` + step number)
3. Average probabilities across all windows from the same record-step
4. Take argmax of averaged probabilities for the record-step prediction

This keeps supervision aligned with the actual labeled signal segment. A record may legitimately contribute a **Severe Step 2** example and a **Mild Step 3** example without creating a contradictory single record label.

In [6]:
# ═══════════════════════════════════════════════════════════════════
# EVALUATION: RECORD-STEP METRICS VIA MEAN PROBABILITY AGGREGATION
# ═══════════════════════════════════════════════════════════════════

def aggregate_to_record_step_level(
    y_proba: np.ndarray,
    metadata: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Aggregate window-level predictions to record-step level via mean probabilities.

    For each unique record-step:
    - Average the softmax probabilities across all its windows
    - Take argmax of the averaged probabilities as the record-step prediction
    - Use the step label attached to that record-step as the true label
    """
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    unique_step_keys = meta['step_key'].unique()
    step_y_true = []
    step_y_pred = []

    for step_key in unique_step_keys:
        mask = meta['step_key'] == step_key
        mean_proba = y_proba[mask].mean(axis=0)
        step_y_pred.append(int(np.argmax(mean_proba)))
        step_y_true.append(int(meta.loc[mask, 'label'].iloc[0]))

    return (
        unique_step_keys,
        np.array(step_y_true, dtype=int),
        np.array(step_y_pred, dtype=int),
    )


def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: List[str] = CLASS_NAMES,
) -> dict:
    """
    Compute comprehensive classification metrics.

    Returns dict with:
    - accuracy, balanced_accuracy, macro_f1
    - per-class precision, recall, f1
    - confusion matrix
    """
    labels = list(range(len(class_names)))

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro',
                        labels=labels, zero_division=0)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    metrics = {
        'accuracy': acc,
        'balanced_accuracy': bal_acc,
        'macro_f1': macro_f1,
        'confusion_matrix': cm,
    }

    for i, name in enumerate(class_names):
        metrics[f'precision_{name}'] = precision[i]
        metrics[f'recall_{name}'] = recall[i]
        metrics[f'f1_{name}'] = f1[i]
        metrics[f'support_{name}'] = int(support[i]) if support[i] is not None else 0

    return metrics


def print_metrics(metrics: dict, prefix: str = "") -> None:
    """Pretty-print aggregated record-step metrics."""
    print(f"\n{prefix}Record-Step Metrics:")
    print(f"  Accuracy:          {metrics['accuracy']:.4f}")
    print(f"  Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
    print(f"  Macro F1:          {metrics['macro_f1']:.4f}")
    print(f"  " + "-" * 45)
    for name in CLASS_NAMES:
        print(f"  {name:8s}  P={metrics[f'precision_{name}']:.3f}  "
              f"R={metrics[f'recall_{name}']:.3f}  "
              f"F1={metrics[f'f1_{name}']:.3f}  "
              f"N={metrics.get(f'support_{name}', '?')}")


print("Evaluation functions defined:")
print("  aggregate_to_record_step_level — mean probability aggregation within each record-step")
print("  compute_metrics                — accuracy, balanced acc, macro F1, per-class")
print("  print_metrics                  — formatted display")


Evaluation functions defined:
  aggregate_to_record_step_level — mean probability aggregation within each record-step
  compute_metrics                — accuracy, balanced acc, macro F1, per-class
  print_metrics                  — formatted display


## 10. Plotting Utilities

In [7]:
# ═══════════════════════════════════════════════════════════════════
# PLOTTING UTILITIES
# ═══════════════════════════════════════════════════════════════════

def plot_confusion_matrix(
    cm: np.ndarray,
    class_names: List[str] = CLASS_NAMES,
    title: str = "Confusion Matrix",
    save_path: Optional[str] = None,
) -> None:
    """Plot a confusion matrix heatmap."""
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, linewidths=0.5,
    )
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_training_history(
    history,
    title: str = "Training History",
    save_path: Optional[str] = None,
) -> None:
    """Plot loss and accuracy curves from training history."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{title} — Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(history.history['accuracy'], label='Train Acc')
    axes[1].plot(history.history['val_accuracy'], label='Val Acc')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'{title} — Accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_fold_summary(
    fold_results: List[dict],
    save_path: Optional[str] = None,
) -> None:
    """Bar chart of per-fold macro F1 with mean ± std."""
    f1s = [r['macro_f1'] for r in fold_results]
    folds = list(range(1, len(f1s) + 1))

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(folds, f1s, color='steelblue', edgecolor='black', alpha=0.8)
    ax.axhline(np.mean(f1s), color='red', linestyle='--',
               label=f'Mean = {np.mean(f1s):.3f} ± {np.std(f1s):.3f}')
    ax.set_xlabel('Fold', fontsize=12)
    ax.set_ylabel('Macro F1', fontsize=12)
    ax.set_title('Per-Fold Record-Level Macro F1', fontsize=14)
    ax.set_xticks(folds)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)

    # Annotate bars
    for bar, val in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=10)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


print("Plotting utilities defined (confusion matrix, training history, fold summary)")

Plotting utilities defined (confusion matrix, training history, fold summary)


## 11. Shared Cross-Validation Utilities

These helpers keep the remaining baseline experiments leakage-safe and record-step aware.

Included here:
1. Stable `step_key` construction for record-step aggregation.
2. One-row-per-record-step target tables for grouped splitting.
3. Constrained grouped CV generation by `rec_id`, with a minimum Severe record-step threshold per fold.

In [8]:
# ═══════════════════════════════════════════════════════════════════
# SHARED GROUPED-CV UTILITIES FOR BASELINES
# ═══════════════════════════════════════════════════════════════════

def _ensure_step_keys(metadata: pd.DataFrame) -> pd.DataFrame:
    """Ensure each window row carries a stable record-step key."""
    meta = metadata.copy()
    if 'step_key' not in meta.columns:
        if 'rec_id' not in meta.columns or 'step' not in meta.columns:
            raise KeyError("metadata must contain 'rec_id' and 'step' columns")
        meta['step_key'] = meta['rec_id'].astype(str) + '__step' + meta['step'].astype(str)
    return meta


def build_step_target_table(metadata: pd.DataFrame) -> pd.DataFrame:
    """Build one labeled row per record-step for grouped splitting and evaluation."""
    meta = _ensure_step_keys(metadata)
    if 'label' not in meta.columns:
        raise KeyError("metadata must contain a 'label' column")

    step_targets = meta[['step_key', 'rec_id', 'step', 'label']].drop_duplicates().copy()
    if step_targets['step_key'].duplicated().any():
        raise RuntimeError("Each step_key must map to exactly one record-step label")

    return step_targets.sort_values(['rec_id', 'step']).reset_index(drop=True)


def stratified_record_kfold_with_constraint(
    metadata: pd.DataFrame,
    n_splits: int = 5,
    min_severe_records: int = 10,
    max_attempts: int = 100,
    cv_seed: int = 42,
    severe_class: int = 2,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Generate grouped CV folds using one labeled row per record-step while
    keeping all steps from the same record in the same fold.

    The config key `min_severe_records` is retained for compatibility, but it now
    refers to the minimum number of Severe record-steps in each validation fold.
    """
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    step_targets = build_step_target_table(meta)

    step_y = step_targets['label'].to_numpy(dtype=int)
    step_groups = step_targets['rec_id'].to_numpy()
    step_keys = step_targets['step_key'].to_numpy()

    total_severe = int(np.sum(step_y == severe_class))
    min_possible = total_severe // n_splits if n_splits > 0 else 0
    print(f"  Total Severe step-units: {total_severe}, {n_splits} folds -> max ~{min_possible}/fold")

    effective_min = min_severe_records
    if effective_min > min_possible:
        if total_severe == 0:
            effective_min = 0
        else:
            effective_min = max(min_possible - 1, 1)
        print(
            f"  WARNING: Relaxed threshold to {effective_min} "
            f"(not enough Severe step-units for {min_severe_records}/fold)"
        )

    best_folds = None
    best_min_severe = -1
    best_counts = None

    n_attempts = max_attempts if HAS_STRATIFIED_GROUP_KFOLD else 1
    for attempt in range(n_attempts):
        seed = cv_seed + attempt

        if HAS_STRATIFIED_GROUP_KFOLD:
            splitter = StratifiedGroupKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=seed,
            )
            split_iter = splitter.split(step_keys, step_y, step_groups)
        else:
            from sklearn.model_selection import GroupKFold
            splitter = GroupKFold(n_splits=n_splits)
            split_iter = splitter.split(step_keys, step_y, step_groups)

        candidate_folds = []
        fold_severe_counts = []
        all_ok = True

        for train_step_idx, val_step_idx in split_iter:
            train_rec_ids = set(step_targets.iloc[train_step_idx]['rec_id'])
            val_rec_ids = set(step_targets.iloc[val_step_idx]['rec_id'])

            n_sev = int(np.sum(step_y[val_step_idx] == severe_class))
            fold_severe_counts.append(n_sev)

            train_mask = meta['rec_id'].isin(train_rec_ids).to_numpy()
            val_mask = meta['rec_id'].isin(val_rec_ids).to_numpy()
            candidate_folds.append((np.where(train_mask)[0], np.where(val_mask)[0]))

            if n_sev < effective_min:
                all_ok = False

        cur_min = min(fold_severe_counts) if fold_severe_counts else -1
        if cur_min > best_min_severe:
            best_min_severe = cur_min
            best_folds = candidate_folds
            best_counts = list(fold_severe_counts)

        if all_ok:
            method = "StratifiedGroupKFold" if HAS_STRATIFIED_GROUP_KFOLD else "GroupKFold"
            print(f"  Found valid split at attempt {attempt + 1} (seed={seed}, {method})")
            print(f"    Severe step-units per val fold: {fold_severe_counts}")
            return candidate_folds

    print(f"  Could not meet >={effective_min} Severe step-units/fold after {n_attempts} attempts.")
    print(f"    Best found: min={best_min_severe}, per-fold={best_counts}")
    return best_folds


print("Shared grouped-CV utilities defined:")
print("  _ensure_step_keys                   - stable record-step keys")
print("  build_step_target_table             - one labeled row per record-step")
print("  stratified_record_kfold_with_constraint - grouped CV with Severe-step constraint")

Shared grouped-CV utilities defined:
  _ensure_step_keys                   - stable record-step keys
  build_step_target_table             - one labeled row per record-step
  stratified_record_kfold_with_constraint - grouped CV with Severe-step constraint


In [14]:
# ═══════════════════════════════════════════════════════════════════
# SHARED TRAINING CALLBACKS FOR RESNET1D BASELINE
# ═══════════════════════════════════════════════════════════════════

class RecordStepBalancedAccuracy(Callback):
    """Compute record-step balanced accuracy at each epoch end."""

    def __init__(
        self,
        X_val: np.ndarray,
        y_val: np.ndarray,
        clin_val: Optional[np.ndarray] = None,
        windows_per_step: int = CONTROLLED_WINDOWS_PER_STEP,
        batch_size: int = 32,
        verbose: bool = True,
        print_every: int = 1,
    ):
        super().__init__()
        self.X_val = X_val
        self.y_val = np.asarray(y_val, dtype=np.int32)
        self.clin_val = clin_val
        self.windows_per_step = int(windows_per_step)
        self.batch_size = batch_size
        self.verbose = bool(verbose)
        self.print_every = max(1, int(print_every))

    def _aggregate_record_step_targets(self, y_proba: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if len(y_proba) != len(self.y_val):
            raise ValueError("Validation probabilities and labels must have the same length")
        if self.windows_per_step <= 0 or len(y_proba) % self.windows_per_step != 0:
            raise ValueError(
                f"Expected validation windows to be divisible by windows_per_step={self.windows_per_step}, got {len(y_proba)}"
            )

        step_y_true = []
        step_y_pred = []
        for start in range(0, len(y_proba), self.windows_per_step):
            end = start + self.windows_per_step
            mean_proba = y_proba[start:end].mean(axis=0)
            y_true_block = self.y_val[start:end]
            if not np.all(y_true_block == y_true_block[0]):
                raise RuntimeError("Validation windows within a record-step block must share the same label")
            step_y_true.append(int(y_true_block[0]))
            step_y_pred.append(int(np.argmax(mean_proba)))

        return np.array(step_y_true, dtype=np.int32), np.array(step_y_pred, dtype=np.int32)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_proba = self.model.predict(self.X_val, batch_size=self.batch_size, verbose=0)
        step_y_true, step_y_pred = self._aggregate_record_step_targets(y_proba)
        bal_acc = float(balanced_accuracy_score(step_y_true, step_y_pred))
        logs['val_record_step_balanced_accuracy'] = bal_acc
        if self.verbose and ((epoch + 1) % self.print_every == 0):
            print(f"  val_record_step_balanced_accuracy: {bal_acc:.4f}")


def get_callbacks(
    patience: int = 20,
    monitor: str = 'val_loss',
    monitor_mode: str = 'min',
    save_path: Optional[str] = None,
    prefix_callbacks: Optional[List[Callback]] = None,
    model_checkpoint_save_weights_only: bool = False,
    callback_verbosity: int = 1,
) -> list:
    callbacks = list(prefix_callbacks or [])
    callbacks.extend([
        EarlyStopping(
            monitor=monitor,
            mode=monitor_mode,
            patience=patience,
            restore_best_weights=True,
            verbose=callback_verbosity,
        ),
        ReduceLROnPlateau(
            monitor=monitor,
            mode=monitor_mode,
            factor=0.5,
            patience=max(patience // 2, 3),
            min_lr=1e-6,
            verbose=callback_verbosity,
        ),
    ])
    if save_path is not None:
        callbacks.append(ModelCheckpoint(
            save_path,
            monitor=monitor,
            mode=monitor_mode,
            save_best_only=True,
            save_weights_only=model_checkpoint_save_weights_only,
            verbose=callback_verbosity,
        ))
    return callbacks


print("Shared ResNet callback utilities defined:")
print("  RecordStepBalancedAccuracy - record-step balanced accuracy callback")
print("  get_callbacks              - early stopping / LR reduction / checkpointing")

Shared ResNet callback utilities defined:
  RecordStepBalancedAccuracy - record-step balanced accuracy callback
  get_callbacks              - early stopping / LR reduction / checkpointing


In [10]:
# ═══════════════════════════════════════════════════════════════════
# METRICS WITH DIAGNOSTIC INDEXES
# ═══════════════════════════════════════════════════════════════════

# Quality index is reported as Youden's J statistic:
#   quality_index = sensitivity + specificity - 1
# This keeps it interpretable for one-vs-rest class evaluation.


def _safe_divide(numerator: float, denominator: float) -> float:
    if denominator <= 0:
        return float('nan')
    return float(numerator / denominator)


def _compute_one_vs_rest_indexes(cm: np.ndarray, class_names: Optional[List[str]] = None) -> dict:
    class_names = class_names or CLASS_NAMES
    cm = np.asarray(cm, dtype=np.float64)
    total = float(cm.sum())
    metrics = {}

    sensitivity_vals = []
    specificity_vals = []
    quality_vals = []

    for class_idx, class_name in enumerate(class_names):
        tp = float(cm[class_idx, class_idx])
        fn = float(cm[class_idx, :].sum() - tp)
        fp = float(cm[:, class_idx].sum() - tp)
        tn = float(total - tp - fn - fp)

        sensitivity = _safe_divide(tp, tp + fn)
        specificity = _safe_divide(tn, tn + fp)
        quality_index = float('nan')
        if not np.isnan(sensitivity) and not np.isnan(specificity):
            quality_index = float(sensitivity + specificity - 1.0)

        metrics[f'sensitivity_{class_name}'] = sensitivity
        metrics[f'specificity_{class_name}'] = specificity
        metrics[f'quality_index_{class_name}'] = quality_index

        sensitivity_vals.append(sensitivity)
        specificity_vals.append(specificity)
        quality_vals.append(quality_index)

    metrics['macro_sensitivity'] = float(np.nanmean(sensitivity_vals))
    metrics['macro_specificity'] = float(np.nanmean(specificity_vals))
    metrics['macro_quality_index'] = float(np.nanmean(quality_vals))
    return metrics


def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: Optional[List[int]] = None,
    class_names: Optional[List[str]] = None,
) -> dict:
    """Compute aggregated record-step metrics with one-vs-rest diagnostic indexes."""
    labels = labels or list(range(NUM_CLASSES))
    class_names = class_names or CLASS_NAMES

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    metrics = {
        'accuracy': float(acc),
        'balanced_accuracy': float(bal_acc),
        'macro_f1': float(macro_f1),
        'confusion_matrix': cm,
    }

    for i, name in enumerate(class_names):
        metrics[f'precision_{name}'] = float(precision[i])
        metrics[f'recall_{name}'] = float(recall[i])
        metrics[f'sensitivity_{name}'] = float(recall[i])
        metrics[f'f1_{name}'] = float(f1[i])
        metrics[f'support_{name}'] = int(support[i]) if support[i] is not None else 0

    metrics.update(_compute_one_vs_rest_indexes(cm, class_names=class_names))
    return metrics


def print_metrics(metrics: dict, prefix: str = "") -> None:
    """Pretty-print aggregated record-step metrics with diagnostic indexes."""
    print(f"\n{prefix}Record-Step Metrics:")
    print(f"  Accuracy:            {metrics['accuracy']:.4f}")
    print(f"  Balanced Accuracy:   {metrics['balanced_accuracy']:.4f}")
    print(f"  Macro F1:            {metrics['macro_f1']:.4f}")
    print(f"  Macro Sensitivity:   {metrics['macro_sensitivity']:.4f}")
    print(f"  Macro Specificity:   {metrics['macro_specificity']:.4f}")
    print(f"  Macro Quality Index: {metrics['macro_quality_index']:.4f}")
    print(f"  " + "-" * 86)
    for name in CLASS_NAMES:
        print(
            f"  {name:8s}  "
            f"P={metrics[f'precision_{name}']:.3f}  "
            f"R={metrics[f'recall_{name}']:.3f}  "
            f"Se={metrics[f'sensitivity_{name}']:.3f}  "
            f"Sp={metrics[f'specificity_{name}']:.3f}  "
            f"F1={metrics[f'f1_{name}']:.3f}  "
            f"QI={metrics[f'quality_index_{name}']:.3f}  "
            f"N={metrics.get(f'support_{name}', '?')}"
        )


print("Metrics utilities active: sensitivity, specificity, and quality index included.")

Metrics utilities active: sensitivity, specificity, and quality index included.


## 12. Baseline Models: Helper Functions

Reusable evaluation functions for the remaining baseline comparison experiments.

**Design principles:**
- Reuse the shared grouped CV folds generated above.
- Reuse `aggregate_to_record_step_level()` for record-step metric computation.
- Reuse `compute_metrics()` for consistent evaluation.
- Standardise features using **training-fold statistics only** (no data leakage).
- For sklearn models: use the engineered feature matrix already defined in the notebook.
- For Keras models (ResNet1D): use the raw signal windows `X_all` directly.
- For sklearn hyperparameter search: use **group-aware inner CV** so all windows from the same `rec_id` stay together during tuning as well as outer evaluation.

In [11]:
# ═══════════════════════════════════════════════════════════════════
# BASELINE HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline

# ── Generate the grouped CV folds used by all remaining baselines ──
baseline_folds = stratified_record_kfold_with_constraint(
    meta_all,
    n_splits=CFG['n_splits'],
    min_severe_records=CFG.get('min_severe_records', 10),
    max_attempts=CFG.get('max_cv_attempts', 100),
    cv_seed=CFG['cv_seed'],
)

# ── Pre-compute engineered features for all windows ─────────────
clin_all_baseline = compute_clinical_features(
    X_all,
    fs=SAMPLING_RATE,
    missing_frac_prefill=meta_all['missing_frac_prefill'].to_numpy(dtype=np.float32),
)
print(f"Engineered feature matrix for baselines: {clin_all_baseline.shape}")
print(f"Number of CV folds: {len(baseline_folds)}")


def make_group_aware_inner_splitter(n_splits: int = 3, random_state: int = GLOBAL_SEED):
    """Create a group-aware splitter for inner hyperparameter tuning."""
    if HAS_STRATIFIED_GROUP_KFOLD:
        return StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state,
        )

    from sklearn.model_selection import GroupKFold
    return GroupKFold(n_splits=n_splits)


def evaluate_sklearn_baseline(
    model_class,
    model_kwargs: dict,
    X_features: np.ndarray,
    y: np.ndarray,
    metadata: pd.DataFrame,
    folds: List[Tuple[np.ndarray, np.ndarray]],
    model_name: str = "Baseline",
    scale_features: bool = True,
    param_distributions: Optional[dict] = None,
    n_search_iter: int = 30,
    inner_cv: int = 3,
) -> List[dict]:
    """
    Evaluate an sklearn classifier fold-by-fold using the same grouped CV,
    the same record-step aggregation logic, and the same metrics.

    If param_distributions is provided, runs a group-aware nested
    RandomizedSearchCV on each outer fold's training split to select the
    best hyperparameters before evaluating on the held-out fold.
    """
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f"\n  {model_name} — Fold {fold_idx + 1}/{len(folds)}")

        X_tr = X_features[train_idx]
        X_va = X_features[val_idx]
        y_tr = y[train_idx]
        y_va = y[val_idx]
        meta_tr = meta.iloc[train_idx].reset_index(drop=True)
        meta_val = meta.iloc[val_idx].reset_index(drop=True)
        groups_tr = meta_tr['rec_id'].astype(str).to_numpy()

        if scale_features:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_va = scaler.transform(X_va)

        if param_distributions is not None:
            inner_splitter = make_group_aware_inner_splitter(
                n_splits=inner_cv,
                random_state=GLOBAL_SEED + fold_idx,
            )
            search = RandomizedSearchCV(
                estimator=model_class(**model_kwargs),
                param_distributions=param_distributions,
                n_iter=n_search_iter,
                cv=inner_splitter,
                scoring='f1_macro',
                random_state=GLOBAL_SEED,
                n_jobs=-1,
                refit=True,
            )
            search.fit(X_tr, y_tr, groups=groups_tr)
            clf = search.best_estimator_
            print(
                f"    Best params: {search.best_params_}  "
                f"(group-aware inner F1={search.best_score_:.4f})"
            )
        else:
            clf = model_class(**model_kwargs)
            clf.fit(X_tr, y_tr)

        y_proba_val = clf.predict_proba(X_va)
        _, step_y_true, step_y_pred = aggregate_to_record_step_level(y_proba_val, meta_val)

        metrics = compute_metrics(step_y_true, step_y_pred)
        metrics['fold'] = fold_idx + 1
        fold_results.append(metrics)

        print(
            f"    Acc={metrics['accuracy']:.4f}  BalAcc={metrics['balanced_accuracy']:.4f}  "
            f"F1={metrics['macro_f1']:.4f}  "
            f"R_N={metrics['recall_Normal']:.3f}  R_M={metrics['recall_Mild']:.3f}  R_S={metrics['recall_Severe']:.3f}"
        )

    return fold_results


def evaluate_keras_baseline(
    build_fn,
    X_signals: np.ndarray,
    y: np.ndarray,
    metadata: pd.DataFrame,
    folds: List[Tuple[np.ndarray, np.ndarray]],
    model_name: str = "KerasBaseline",
    epochs: int = 100,
    batch_size: int = 16,
    patience: int = 20,
    lr: float = 1e-3,
    use_class_weights: bool = True,
    augment: bool = True,
    label_smoothing: float = 0.05,
) -> List[dict]:
    """
    Evaluate a Keras model fold-by-fold using the same grouped CV,
    the same record-step aggregation logic, and the same metrics.
    """
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f"\n  {model_name} — Fold {fold_idx + 1}/{len(folds)}")

        X_tr = X_signals[train_idx]
        X_va = X_signals[val_idx]
        y_tr = y[train_idx]
        y_va = y[val_idx]
        meta_val = meta.iloc[val_idx].reset_index(drop=True)

        norm_stats = compute_norm_stats(X_tr)
        X_tr_n = apply_norm(X_tr, norm_stats)
        X_va_n = apply_norm(X_va, norm_stats)

        if augment:
            X_tr_n = augment_signal(X_tr_n, CFG)

        class_weight = None
        if use_class_weights:
            classes = np.unique(y_tr)
            weights = compute_class_weight('balanced', classes=classes, y=y_tr)
            class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

        y_tr_cat = to_categorical(y_tr, NUM_CLASSES)
        y_va_cat = to_categorical(y_va, NUM_CLASSES)

        tf.keras.backend.clear_session()
        model = build_fn()

        model.compile(
            optimizer=Adam(learning_rate=lr),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
            metrics=['accuracy'],
        )

        rs_callback = RecordStepBalancedAccuracy(
            X_val=X_va_n,
            y_val=y_va,
            windows_per_step=CONTROLLED_WINDOWS_PER_STEP,
            batch_size=32,
        )

        callbacks = get_callbacks(
            patience=patience,
            monitor='val_record_step_balanced_accuracy',
            monitor_mode='max',
            prefix_callbacks=[rs_callback],
            model_checkpoint_save_weights_only=True,
        )

        model.fit(
            X_tr_n,
            y_tr_cat,
            validation_data=(X_va_n, y_va_cat),
            epochs=epochs,
            batch_size=batch_size,
            class_weight=class_weight,
            callbacks=callbacks,
            verbose=0,
        )

        y_proba_val = model.predict(X_va_n, batch_size=32, verbose=0)
        _, step_y_true, step_y_pred = aggregate_to_record_step_level(y_proba_val, meta_val)

        metrics = compute_metrics(step_y_true, step_y_pred)
        metrics['fold'] = fold_idx + 1
        fold_results.append(metrics)

        print(
            f"    Acc={metrics['accuracy']:.4f}  BalAcc={metrics['balanced_accuracy']:.4f}  "
            f"F1={metrics['macro_f1']:.4f}  "
            f"R_N={metrics['recall_Normal']:.3f}  R_M={metrics['recall_Mild']:.3f}  R_S={metrics['recall_Severe']:.3f}"
        )

        del model
        tf.keras.backend.clear_session()

    return fold_results


def summarise_fold_results(fold_results: List[dict], model_name: str) -> dict:
    """Summarise fold results into mean ± std row for the comparison table."""
    summary = {'model': model_name}
    for key in ['accuracy', 'balanced_accuracy', 'macro_f1']:
        vals = [r[key] for r in fold_results]
        summary[f'{key}_mean'] = float(np.mean(vals))
        summary[f'{key}_std'] = float(np.std(vals))
    for name in CLASS_NAMES:
        vals = [r[f'recall_{name}'] for r in fold_results]
        summary[f'{name.lower()}_recall_mean'] = float(np.mean(vals))
        summary[f'{name.lower()}_recall_std'] = float(np.std(vals))
    return summary


print("Baseline helper functions defined:")
print("  make_group_aware_inner_splitter - group-aware inner CV for sklearn tuning")
print("  evaluate_sklearn_baseline       - fold-by-fold sklearn evaluation (group-aware nested HP search)")
print("  evaluate_keras_baseline         - fold-by-fold Keras evaluation")
print("  summarise_fold_results          - mean ± std summary row")
print(f"  baseline_folds                  - {len(baseline_folds)} grouped folds")
print(f"  clin_all_baseline               - {clin_all_baseline.shape} engineered feature matrix")

  Total Severe step-units: 66, 10 folds -> max ~6/fold
  Found valid split at attempt 8 (seed=49, StratifiedGroupKFold)
    Severe step-units per val fold: [7, 7, 7, 8, 5, 6, 5, 6, 10, 5]
Engineered feature matrix for baselines: (1398, 10)
Number of CV folds: 10
Baseline helper functions defined:
  make_group_aware_inner_splitter - group-aware inner CV for sklearn tuning
  evaluate_sklearn_baseline       - fold-by-fold sklearn evaluation (group-aware nested HP search)
  evaluate_keras_baseline         - fold-by-fold Keras evaluation
  summarise_fold_results          - mean ± std summary row
  baseline_folds                  - 10 grouped folds
  clin_all_baseline               - (1398, 10) engineered feature matrix


## 13. Baseline Models: Feature-Based Classical ML (Tuned via Group-Aware Nested CV)

Three classical baselines using the **same engineered features** extracted by `compute_clinical_features()`.

**Hyperparameter tuning strategy: Group-Aware Nested Cross-Validation**
- **Outer loop**: Same grouped CV folds as the other baselines, so the comparison is fold-matched and leakage-safe.
- **Inner loop**: Group-aware `RandomizedSearchCV` on each outer fold's training split, keeping all windows from the same `rec_id` together during tuning.
- The best hyperparameters are selected *per outer fold*, and the tuned model is evaluated on the held-out outer fold.

**Search spaces**

| Model | Features | Scaling | Search space |
|-------|----------|---------|-------------|
| **Logistic Regression** | 10 engineered | StandardScaler | `C` ∈ [1e-3, 1e3] (log), valid multinomial combinations only: (`l1`, `saga`), (`l2`, `lbfgs` or `saga`), (`elasticnet`, `saga`), with `l1_ratio` ∈ [0.1, 0.9] for elasticnet only |
| **SVM** | 10 engineered | StandardScaler | `C` ∈ [1e-2, 1e3] (log), `gamma` ∈ {scale, auto, 1e-4, 1e-3, 1e-2, 1e-1, 1}, `kernel` ∈ {rbf, linear, poly}, `degree` ∈ {2, 3} (if poly) |
| **Random Forest** | 10 engineered | None | `n_estimators` ∈ {100, 200, 300, 500, 800}, `max_depth` ∈ {None, 10, 20, 30, 50, 70}, `min_samples_split` ∈ {2, 5, 10}, `min_samples_leaf` ∈ {1, 2, 4}, `max_features` ∈ {sqrt, log2, None, 0.5} |

In [12]:
# ═══════════════════════════════════════════════════════════════════
# BASELINE A: MULTINOMIAL LOGISTIC REGRESSION (TUNED — WIDENED)
# ═══════════════════════════════════════════════════════════════════
from scipy.stats import loguniform, uniform

print("\n" + "#" * 70)
print("# BASELINE: Logistic Regression (Nested RandomizedSearchCV — Widened)")
print("#" * 70)

# Use conditional search spaces so every sampled configuration is valid for
# multinomial logistic regression. This avoids wasting inner-loop trials on
# incompatible solver / penalty combinations.
lr_param_dist = [
    {
        'C': loguniform(1e-3, 1e3),
        'penalty': ['l1'],
        'solver': ['saga'],
    },
    {
        'C': loguniform(1e-3, 1e3),
        'penalty': ['l2'],
        'solver': ['lbfgs', 'saga'],
    },
    {
        'C': loguniform(1e-3, 1e3),
        'penalty': ['elasticnet'],
        'solver': ['saga'],
        'l1_ratio': uniform(0.1, 0.8),
    },
]

lr_results = evaluate_sklearn_baseline(
    model_class=LogisticRegression,
    model_kwargs=dict(
        multi_class='multinomial',
        max_iter=3000,
        class_weight='balanced',
        random_state=GLOBAL_SEED,
    ),
    X_features=clin_all_baseline,
    y=y_all,
    metadata=meta_all,
    folds=baseline_folds,
    model_name="Logistic Regression",
    scale_features=True,
    param_distributions=lr_param_dist,
    n_search_iter=40,
    inner_cv=3,
)

lr_summary = summarise_fold_results(lr_results, "Logistic Regression")
print(f"\nLogistic Regression — Mean Macro F1: {lr_summary['macro_f1_mean']:.4f} ± {lr_summary['macro_f1_std']:.4f}")
print(f"  Balanced Accuracy: {lr_summary['balanced_accuracy_mean']:.4f} ± {lr_summary['balanced_accuracy_std']:.4f}")


# ═══════════════════════════════════════════════════════════════════
# BASELINE B: SVM (TUNED — WIDENED)
# ═══════════════════════════════════════════════════════════════════

print("\n" + "#" * 70)
print("# BASELINE: SVM (Nested RandomizedSearchCV — Widened)")
print("#" * 70)

# C upper bound widened from 1e2 → 1e3.
# Added poly kernel + degree search, expanded gamma range.
svm_param_dist = {
    'C': loguniform(1e-2, 1e3),
    'kernel': ['rbf', 'linear', 'poly'],
    'gamma': ['scale', 'auto', 1e-4, 1e-3, 1e-2, 1e-1, 1],
    'degree': [2, 3],               # only used when kernel='poly'
}

svm_results = evaluate_sklearn_baseline(
    model_class=SVC,
    model_kwargs=dict(
        probability=True,
        class_weight='balanced',
        random_state=GLOBAL_SEED,
    ),
    X_features=clin_all_baseline,
    y=y_all,
    metadata=meta_all,
    folds=baseline_folds,
    model_name="SVM",
    scale_features=True,
    param_distributions=svm_param_dist,
    n_search_iter=50,
    inner_cv=3,
)

svm_summary = summarise_fold_results(svm_results, "SVM")
print(f"\nSVM — Mean Macro F1: {svm_summary['macro_f1_mean']:.4f} ± {svm_summary['macro_f1_std']:.4f}")
print(f"  Balanced Accuracy: {svm_summary['balanced_accuracy_mean']:.4f} ± {svm_summary['balanced_accuracy_std']:.4f}")


# ═══════════════════════════════════════════════════════════════════
# BASELINE C: RANDOM FOREST (TUNED — WIDENED)
# ═══════════════════════════════════════════════════════════════════

print("\n" + "#" * 70)
print("# BASELINE: Random Forest (Nested RandomizedSearchCV — Widened)")
print("#" * 70)

# max_depth widened from [None,10,20,30] → [None,10,20,30,50,70] (was saturating at 30).
# Added max_features search. Bumped n_estimators to include 800.
rf_param_dist = {
    'n_estimators': [100, 200, 300, 500, 800],
    'max_depth': [None, 10, 20, 30, 50, 70],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None, 0.5],
}

rf_results = evaluate_sklearn_baseline(
    model_class=RandomForestClassifier,
    model_kwargs=dict(
        class_weight='balanced_subsample',
        random_state=GLOBAL_SEED,
        n_jobs=-1,
    ),
    X_features=clin_all_baseline,
    y=y_all,
    metadata=meta_all,
    folds=baseline_folds,
    model_name="Random Forest",
    scale_features=False,
    param_distributions=rf_param_dist,
    n_search_iter=60,
    inner_cv=3,
)

rf_summary = summarise_fold_results(rf_results, "Random Forest")
print(f"\nRandom Forest — Mean Macro F1: {rf_summary['macro_f1_mean']:.4f} ± {rf_summary['macro_f1_std']:.4f}")
print(f"  Balanced Accuracy: {rf_summary['balanced_accuracy_mean']:.4f} ± {rf_summary['balanced_accuracy_std']:.4f}")


# ═══════════════════════════════════════════════════════════════════
# CLASSICAL ML SUMMARY
# ═══════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("CLASSICAL ML BASELINES — SUMMARY (Tuned — Widened, Record-Step Level)")
print("=" * 70)

for name, results in [("Logistic Regression", lr_results),
                       ("SVM", svm_results),
                       ("Random Forest", rf_results)]:
    accs = [r['accuracy'] for r in results]
    bals = [r['balanced_accuracy'] for r in results]
    f1s = [r['macro_f1'] for r in results]
    print(f"\n  {name}:")
    print(f"    Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"    Balanced Accuracy: {np.mean(bals):.4f} ± {np.std(bals):.4f}")
    print(f"    Macro F1:          {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    for cls_name in CLASS_NAMES:
        recs = [r[f'recall_{cls_name}'] for r in results]
        print(f"    Recall {cls_name:8s}:  {np.mean(recs):.4f} ± {np.std(recs):.4f}")


######################################################################
# BASELINE: Logistic Regression (Nested RandomizedSearchCV — Widened)
######################################################################

  Logistic Regression — Fold 1/10
    Best params: {'C': 70.85721663941601, 'penalty': 'l2', 'solver': 'lbfgs'}  (group-aware inner F1=0.4539)
    Acc=0.5417  BalAcc=0.6264  F1=0.5452  R_N=0.647  R_M=0.375  R_S=0.857

  Logistic Regression — Fold 2/10
    Best params: {'C': 1.40779231399724, 'penalty': 'l2', 'solver': 'saga'}  (group-aware inner F1=0.4569)
    Acc=0.6154  BalAcc=0.6220  F1=0.5859  R_N=0.731  R_M=0.421  R_S=0.714

  Logistic Regression — Fold 3/10
    Best params: {'C': 0.19713872690045356, 'l1_ratio': 0.8865847086454306, 'penalty': 'elasticnet', 'solver': 'saga'}  (group-aware inner F1=0.4673)
    Acc=0.4524  BalAcc=0.4499  F1=0.4339  R_N=0.500  R_M=0.421  R_S=0.429

  Logistic Regression — Fold 4/10
    Best params: {'C': 0.4994877024461299, 'penalty': 'l2',

## 14. Baseline Model: ResNet1D (Grouped Multi-Split Search)

A compact 1D residual network for direct time-series classification, with a **stronger but still practical** hyperparameter search over a small architecture/training space.

**Tuning strategy:**
- Define a modest search space over learning rate, dropout, kernel size, and base filters.
- Build a separate set of **grouped tuning folds** for model selection.
- Evaluate each candidate across **multiple grouped tuning splits** and rank by mean record-step Macro F1.
- Then run the selected configuration across the main outer grouped CV folds for final reporting.

**Why this is stronger than the previous version:**
- Selection no longer depends on a single fold.
- Tuning remains group-aware at the record level.
- The search space is widened slightly without turning this into a full nested deep-learning optimization.

**Uses raw signal windows only** — no handcrafted features.
Same outer folds, same augmentation, same callbacks, same class weighting, and same record-step aggregation as the other baseline models.

**Note:** this remains a practical model-selection stage rather than full nested deep tuning.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# RESNET1D BUILDER (PARAMETERISED FOR TUNING)
# ═══════════════════════════════════════════════════════════════════

def _resnet1d_block(x, filters: int, kernel_size: int = 7, name: str = "res"):
    """
    A single 1D residual block: Conv→BN→ReLU→Conv→BN + skip → ReLU.
    Uses a 1x1 projection shortcut if the filter count changes.
    """
    shortcut = x

    if int(x.shape[-1]) != filters:
        shortcut = Conv1D(filters, 1, padding='same', use_bias=False,
                          name=f"{name}_proj")(shortcut)
        shortcut = BatchNormalization(name=f"{name}_proj_bn")(shortcut)

    x = Conv1D(filters, kernel_size, padding='same', use_bias=False,
               name=f"{name}_conv1")(x)
    x = BatchNormalization(name=f"{name}_bn1")(x)
    x = Activation('relu', name=f"{name}_relu1")(x)

    x = Conv1D(filters, kernel_size, padding='same', use_bias=False,
               name=f"{name}_conv2")(x)
    x = BatchNormalization(name=f"{name}_bn2")(x)

    x = Add(name=f"{name}_add")([x, shortcut])
    x = Activation('relu', name=f"{name}_relu2")(x)
    return x


def build_resnet1d(
    input_length: int = SEGMENT_LENGTH,
    num_classes: int = NUM_CLASSES,
    dropout_rate: float = 0.40,
    kernel_size: int = 7,
    base_filters: int = 64,
) -> Model:
    """
    Build a compact ResNet1D for 1D time-series classification.

    Architecture:
        Input → Conv1D(base_filters, kernel_size) → BN → ReLU → MaxPool
        → ResBlock(base_filters) → MaxPool
        → ResBlock(base_filters*2) → MaxPool
        → ResBlock(base_filters*2)
        → GlobalAvgPool → Dropout → Dense(softmax)
    """
    inputs = Input(shape=(input_length, 1), name="resnet1d_input")

    x = Conv1D(base_filters, kernel_size, padding='same', use_bias=False,
               name="resnet1d_init_conv")(inputs)
    x = BatchNormalization(name="resnet1d_init_bn")(x)
    x = Activation('relu', name="resnet1d_init_relu")(x)
    x = layers.MaxPooling1D(pool_size=4, name="resnet1d_init_pool")(x)

    x = _resnet1d_block(x, filters=base_filters, kernel_size=kernel_size,
                        name="resnet1d_blk1")
    x = layers.MaxPooling1D(pool_size=4, name="resnet1d_pool1")(x)

    x = _resnet1d_block(x, filters=base_filters * 2, kernel_size=kernel_size,
                        name="resnet1d_blk2")
    x = layers.MaxPooling1D(pool_size=4, name="resnet1d_pool2")(x)

    x = _resnet1d_block(x, filters=base_filters * 2, kernel_size=kernel_size,
                        name="resnet1d_blk3")

    x = GlobalAveragePooling1D(name="resnet1d_gap")(x)
    x = Dropout(dropout_rate, name="resnet1d_drop")(x)
    outputs = Dense(num_classes, activation='softmax', name="resnet1d_output")(x)

    model = Model(inputs, outputs, name="ResNet1D")
    return model


def fit_model_with_batch_backoff(
    model_builder,
    compile_kwargs: dict,
    fit_kwargs: dict,
    initial_batch_size: int,
    min_batch_size: int = 1,
) -> Tuple[Model, object, int]:
    """Train a model, retrying with smaller batches if TensorFlow runs out of memory."""
    batch_size = max(int(initial_batch_size), int(min_batch_size))
    last_error = None

    while batch_size >= min_batch_size:
        tf.keras.backend.clear_session()
        gc.collect()
        model = model_builder()
        model.compile(**compile_kwargs)

        try:
            history = model.fit(batch_size=batch_size, **fit_kwargs)
            return model, history, batch_size
        except tf.errors.ResourceExhaustedError as exc:
            last_error = exc
            print(f"      OOM at batch_size={batch_size}; retrying with a smaller batch.")
            del model
            tf.keras.backend.clear_session()
            gc.collect()

            if batch_size == min_batch_size:
                break

            next_batch_size = max(min_batch_size, batch_size // 2)
            if next_batch_size == batch_size:
                next_batch_size = batch_size - 1
            batch_size = next_batch_size

    raise last_error


def evaluate_resnet_config(
    cfg_hp: dict,
    folds: List[Tuple[np.ndarray, np.ndarray]],
    model_name: str = "ResNet1D tuning",
) -> dict:
    """Evaluate one ResNet config across grouped tuning folds."""
    meta = _ensure_step_keys(meta_all.reset_index(drop=True))
    fold_scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f"\n    {model_name} — tuning fold {fold_idx + 1}/{len(folds)}")

        X_tr = X_all[train_idx]
        X_va = X_all[val_idx]
        y_tr = y_all[train_idx]
        y_va = y_all[val_idx]
        meta_val = meta.iloc[val_idx].reset_index(drop=True)

        norm_stats = compute_norm_stats(X_tr)
        X_tr_n = apply_norm(X_tr, norm_stats)
        X_va_n = apply_norm(X_va, norm_stats)

        if CFG.get('augment_train', True):
            X_tr_n = augment_signal(X_tr_n, CFG)

        classes = np.unique(y_tr)
        weights = compute_class_weight('balanced', classes=classes, y=y_tr)
        class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

        y_tr_cat = to_categorical(y_tr, NUM_CLASSES)
        y_va_cat = to_categorical(y_va, NUM_CLASSES)

        model_builder = lambda: build_resnet1d(
            dropout_rate=float(cfg_hp['dropout_rate']),
            kernel_size=int(cfg_hp['kernel_size']),
            base_filters=int(cfg_hp['base_filters']),
        )
        compile_kwargs = dict(
            optimizer=Adam(learning_rate=float(cfg_hp['lr'])),
            loss=tf.keras.losses.CategoricalCrossentropy(
                label_smoothing=CFG.get('label_smoothing', 0.05)
            ),
            metrics=['accuracy'],
        )

        rs_cb = RecordStepBalancedAccuracy(
            X_val=X_va_n,
            y_val=y_va,
            windows_per_step=CONTROLLED_WINDOWS_PER_STEP,
            batch_size=32,
            verbose=False,
        )
        cbs = get_callbacks(
            patience=CFG['scratch_patience'],
            monitor='val_record_step_balanced_accuracy',
            monitor_mode='max',
            prefix_callbacks=[rs_cb],
            model_checkpoint_save_weights_only=True,
            callback_verbosity=0,
        )
        fit_kwargs = dict(
            x=X_tr_n,
            y=y_tr_cat,
            validation_data=(X_va_n, y_va_cat),
            epochs=CFG['scratch_epochs'],
            class_weight=class_weight,
            callbacks=cbs,
            verbose=0,
        )

        try:
            model, _, used_batch_size = fit_model_with_batch_backoff(
                model_builder=model_builder,
                compile_kwargs=compile_kwargs,
                fit_kwargs=fit_kwargs,
                initial_batch_size=int(CFG['scratch_batch']),
            )
        except tf.errors.ResourceExhaustedError:
            print(f"      OOM persisted for fold {fold_idx + 1}; marking this config as failed.")
            return {
                **cfg_hp,
                'macro_f1_mean': float('-inf'),
                'macro_f1_std': float('nan'),
                'balanced_accuracy_mean': float('-inf'),
                'balanced_accuracy_std': float('nan'),
                'status': 'oom',
                'failed_fold': fold_idx + 1,
            }

        y_proba = model.predict(X_va_n, batch_size=32, verbose=0)
        _, step_y_true, step_y_pred = aggregate_to_record_step_level(y_proba, meta_val)
        metrics = compute_metrics(step_y_true, step_y_pred)
        fold_scores.append(metrics)

        print(
            f"      batch={used_batch_size}  "
            f"BalAcc={metrics['balanced_accuracy']:.4f}  "
            f"F1={metrics['macro_f1']:.4f}"
        )

        del model
        tf.keras.backend.clear_session()
        gc.collect()

    macro_f1_vals = [m['macro_f1'] for m in fold_scores]
    bal_acc_vals = [m['balanced_accuracy'] for m in fold_scores]
    return {
        **cfg_hp,
        'macro_f1_mean': float(np.mean(macro_f1_vals)),
        'macro_f1_std': float(np.std(macro_f1_vals)),
        'balanced_accuracy_mean': float(np.mean(bal_acc_vals)),
        'balanced_accuracy_std': float(np.std(bal_acc_vals)),
        'status': 'ok',
    }


def evaluate_resnet_outer_folds(
    build_fn,
    X_signals: np.ndarray,
    y: np.ndarray,
    metadata: pd.DataFrame,
    folds: List[Tuple[np.ndarray, np.ndarray]],
    model_name: str = "ResNet1D",
    epochs: int = 100,
    batch_size: int = 16,
    patience: int = 20,
    lr: float = 1e-3,
    use_class_weights: bool = True,
    augment: bool = True,
    label_smoothing: float = 0.05,
) -> List[dict]:
    """Evaluate a ResNet baseline fold-by-fold with OOM-aware batch backoff."""
    meta = _ensure_step_keys(metadata.reset_index(drop=True))
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f"\n  {model_name} — Fold {fold_idx + 1}/{len(folds)}")

        X_tr = X_signals[train_idx]
        X_va = X_signals[val_idx]
        y_tr = y[train_idx]
        y_va = y[val_idx]
        meta_val = meta.iloc[val_idx].reset_index(drop=True)

        norm_stats = compute_norm_stats(X_tr)
        X_tr_n = apply_norm(X_tr, norm_stats)
        X_va_n = apply_norm(X_va, norm_stats)

        if augment:
            X_tr_n = augment_signal(X_tr_n, CFG)

        class_weight = None
        if use_class_weights:
            classes = np.unique(y_tr)
            weights = compute_class_weight('balanced', classes=classes, y=y_tr)
            class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

        y_tr_cat = to_categorical(y_tr, NUM_CLASSES)
        y_va_cat = to_categorical(y_va, NUM_CLASSES)

        compile_kwargs = dict(
            optimizer=Adam(learning_rate=lr),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
            metrics=['accuracy'],
        )
        rs_callback = RecordStepBalancedAccuracy(
            X_val=X_va_n,
            y_val=y_va,
            windows_per_step=CONTROLLED_WINDOWS_PER_STEP,
            batch_size=32,
            verbose=False,
        )
        callbacks = get_callbacks(
            patience=patience,
            monitor='val_record_step_balanced_accuracy',
            monitor_mode='max',
            prefix_callbacks=[rs_callback],
            model_checkpoint_save_weights_only=True,
            callback_verbosity=0,
        )
        fit_kwargs = dict(
            x=X_tr_n,
            y=y_tr_cat,
            validation_data=(X_va_n, y_va_cat),
            epochs=epochs,
            class_weight=class_weight,
            callbacks=callbacks,
            verbose=0,
        )

        model, _, used_batch_size = fit_model_with_batch_backoff(
            model_builder=build_fn,
            compile_kwargs=compile_kwargs,
            fit_kwargs=fit_kwargs,
            initial_batch_size=int(batch_size),
        )

        y_proba_val = model.predict(X_va_n, batch_size=32, verbose=0)
        _, step_y_true, step_y_pred = aggregate_to_record_step_level(y_proba_val, meta_val)

        metrics = compute_metrics(step_y_true, step_y_pred)
        metrics['fold'] = fold_idx + 1
        fold_results.append(metrics)

        print(
            f"    batch={used_batch_size}  "
            f"Acc={metrics['accuracy']:.4f}  BalAcc={metrics['balanced_accuracy']:.4f}  "
            f"F1={metrics['macro_f1']:.4f}  "
            f"R_N={metrics['recall_Normal']:.3f}  R_M={metrics['recall_Mild']:.3f}  R_S={metrics['recall_Severe']:.3f}"
        )

        del model
        tf.keras.backend.clear_session()
        gc.collect()

    return fold_results


# Quick architecture check with defaults
_test_resnet = build_resnet1d()
_test_resnet.summary()
print(f"\nResNet1D (default) parameters: {_test_resnet.count_params():,}")
del _test_resnet


# ═══════════════════════════════════════════════════════════════════
# RESNET1D GROUPED MULTI-SPLIT HP SEARCH
# ═══════════════════════════════════════════════════════════════════
# Practical model-selection stage: grouped tuning folds + slightly wider grid.

import itertools

RESNET_TUNING_SPLITS = 3
RESNET_TUNING_CV_SEED = CFG['cv_seed'] + 1000

resnet_search_space = {
    'lr': [3e-4, 5e-4, 1e-3],
    'dropout_rate': [0.30, 0.40, 0.50],
    'kernel_size': [5, 9],
    'base_filters': [32, 64],
}

resnet_configs = [
    dict(zip(resnet_search_space.keys(), values))
    for values in itertools.product(*resnet_search_space.values())
]

resnet_tuning_folds = stratified_record_kfold_with_constraint(
    meta_all,
    n_splits=RESNET_TUNING_SPLITS,
    min_severe_records=max(1, CFG.get('min_severe_records', 10)),
    max_attempts=CFG.get('max_cv_attempts', 100),
    cv_seed=RESNET_TUNING_CV_SEED,
)

print(f"\n{'#' * 70}")
print(f"# ResNet1D grouped tuning search: {len(resnet_configs)} configs × {len(resnet_tuning_folds)} grouped tuning folds")
print(f"{'#' * 70}")

config_scores = []
for cfg_idx, cfg_hp in enumerate(resnet_configs):
    print(f"\n  Config {cfg_idx + 1}/{len(resnet_configs)}: {cfg_hp}")
    config_result = evaluate_resnet_config(
        cfg_hp,
        resnet_tuning_folds,
        model_name="ResNet1D grouped search",
    )
    config_scores.append(config_result)
    status = config_result.get('status', 'ok')
    if status != 'ok':
        print(f"    → Skipped after {status} on fold {config_result.get('failed_fold', '?')}")
        continue
    print(
        f"    → Mean BalAcc={config_result['balanced_accuracy_mean']:.4f} ± {config_result['balanced_accuracy_std']:.4f}  "
        f"Mean F1={config_result['macro_f1_mean']:.4f} ± {config_result['macro_f1_std']:.4f}"
    )

config_scores_df = pd.DataFrame(config_scores)
valid_config_scores_df = config_scores_df[config_scores_df['status'] == 'ok'].copy()
if valid_config_scores_df.empty:
    raise RuntimeError("All ResNet1D tuning configs failed. Reduce model size or training batch size.")

valid_config_scores_df = valid_config_scores_df.sort_values(
    ['macro_f1_mean', 'balanced_accuracy_mean'],
    ascending=False,
).reset_index(drop=True)
best_cfg = valid_config_scores_df.iloc[0]

print(f"\n{'=' * 70}")
print("Best ResNet1D config (grouped multi-split search):")
print(
    f"  lr={best_cfg['lr']}, dropout={best_cfg['dropout_rate']}, "
    f"kernel_size={int(best_cfg['kernel_size'])}, base_filters={int(best_cfg['base_filters'])}"
)
print(
    f"  Mean tuning Macro F1={best_cfg['macro_f1_mean']:.4f} ± {best_cfg['macro_f1_std']:.4f}  "
    f"BalAcc={best_cfg['balanced_accuracy_mean']:.4f} ± {best_cfg['balanced_accuracy_std']:.4f}"
)
print(f"{'=' * 70}")

config_scores_df.to_csv(OUTPUT_DIR / "resnet1d_hp_search.csv", index=False)


# ═══════════════════════════════════════════════════════════════════
# FINAL OUTER-FOLD EVALUATION WITH THE SELECTED CONFIG
# ═══════════════════════════════════════════════════════════════════

print(f"\n{'#' * 70}")
print(f"# ResNet1D — selected config across all {len(baseline_folds)} outer folds")
print(f"{'#' * 70}")

best_build_fn = lambda: build_resnet1d(
    dropout_rate=float(best_cfg['dropout_rate']),
    kernel_size=int(best_cfg['kernel_size']),
    base_filters=int(best_cfg['base_filters']),
)

resnet_results = evaluate_resnet_outer_folds(
    build_fn=best_build_fn,
    X_signals=X_all,
    y=y_all,
    metadata=meta_all,
    folds=baseline_folds,
    model_name="ResNet1D (grouped-tuned)",
    epochs=CFG['scratch_epochs'],
    batch_size=CFG['scratch_batch'],
    patience=CFG['scratch_patience'],
    lr=float(best_cfg['lr']),
    use_class_weights=True,
    augment=CFG.get('augment_train', True),
    label_smoothing=CFG.get('label_smoothing', 0.05),
)

resnet_summary = summarise_fold_results(resnet_results, "ResNet1D (grouped-tuned)")
print(f"\nResNet1D (grouped-tuned) — Mean Macro F1: {resnet_summary['macro_f1_mean']:.4f} ± {resnet_summary['macro_f1_std']:.4f}")
print(f"  Balanced Accuracy: {resnet_summary['balanced_accuracy_mean']:.4f} ± {resnet_summary['balanced_accuracy_std']:.4f}")
print(f"  Per-class recall:  Normal={resnet_summary['normal_recall_mean']:.4f}  "
      f"Mild={resnet_summary['mild_recall_mean']:.4f}  "
      f"Severe={resnet_summary['severe_recall_mean']:.4f}")

Model: "ResNet1D"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ resnet1d_input      │ (None, 1800, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_init_conv  │ (None, 1800, 64)  │        448 │ resnet1d_input[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_init_bn    │ (None, 1800, 64)  │        256 │ resnet1d_init_co… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_init_relu  │ (None, 1800, 64)  │          0 │ resnet1d_init_bn… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_init_pool  │ (None, 450, 64)   │          0 │ resnet1d_init_re… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk1_conv1 │ (None, 450, 64)   │     28,672 │ resnet1d_init_po… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk1_bn1   │ (None, 450, 64)   │        256 │ resnet1d_blk1_co… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk1_relu1 │ (None, 450, 64)   │          0 │ resnet1d_blk1_bn… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk1_conv2 │ (None, 450, 64)   │     28,672 │ resnet1d_blk1_re… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk1_bn2   │ (None, 450, 64)   │        256 │ resnet1d_blk1_co… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk1_add   │ (None, 450, 64)   │          0 │ resnet1d_blk1_bn… │
│ (Add)               │                   │            │ resnet1d_init_po… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk1_relu2 │ (None, 450, 64)   │          0 │ resnet1d_blk1_ad… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_pool1      │ (None, 112, 64)   │          0 │ resnet1d_blk1_re… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk2_conv1 │ (None, 112, 128)  │     57,344 │ resnet1d_pool1[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk2_bn1   │ (None, 112, 128)  │        512 │ resnet1d_blk2_co… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk2_relu1 │ (None, 112, 128)  │          0 │ resnet1d_blk2_bn… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet1d_blk2_conv2 │ (None, 112, 128)  │    114,688 │ resnet1d_blk2_re

 Total params: 471,107 (1.80 MB)

 Trainable params: 469,443 (1.79 MB)

 Non-trainable params: 1,664 (6.50 KB)


ResNet1D (default) parameters: 471,107
  Total Severe step-units: 66, 3 folds -> max ~22/fold
  Found valid split at attempt 1 (seed=1042, StratifiedGroupKFold)
    Severe step-units per val fold: [20, 28, 18]

######################################################################
# ResNet1D grouped tuning search: 36 configs × 3 grouped tuning folds
######################################################################

  Config 1/36: {'lr': 0.0003, 'dropout_rate': 0.3, 'kernel_size': 5, 'base_filters': 32}

    ResNet1D grouped search — tuning fold 1/3

  val_record_step_balanced_accuracy: 0.4660
  val_record_step_balanced_accuracy: 0.4763
  val_record_step_balanced_accuracy: 0.4285
  val_record_step_balanced_accuracy: 0.5402
  val_record_step_balanced_accuracy: 0.5099
  val_record_step_balanced_accuracy: 0.5268
  val_record_step_balanced_accuracy: 0.5112
  val_record_step_balanced_accuracy: 0.5037
  val_record_step_balanced_accuracy: 0.4597
  val_record_step_balanced_accuracy: 0.534

ResourceExhaustedError: Graph execution error:

Detected at node gradient_tape/ResNet1D_1/resnet1d_blk1_conv2_1/convolution/Conv2DBackpropFilter defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "c:\Users\ZEYNEP\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "C:\Users\ZEYNEP\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance

  File "c:\Users\ZEYNEP\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start

  File "C:\Users\ZEYNEP\AppData\Roaming\Python\Python311\site-packages\tornado\platform\asyncio.py", line 211, in start

  File "c:\Users\ZEYNEP\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 604, in run_forever

  File "c:\Users\ZEYNEP\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_

## 15. Model Comparison

Final side-by-side comparison of the remaining models evaluated on the same dataset, same grouped outer cross-validation folds, and same record-step aggregation logic. Classical ML baselines are **tuned via group-aware nested RandomizedSearchCV** on each outer training split. ResNet1D is tuned via a **grouped multi-split model-selection stage** and then evaluated across the main outer folds.

| Model | Input | HP Tuning | Type |
|-------|-------|-----------|------|
| Logistic Regression | 10 engineered features | Group-aware nested RandomizedSearchCV (40 iter) | Classical ML |
| SVM | 10 engineered features | Group-aware nested RandomizedSearchCV (50 iter) | Classical ML |
| Random Forest | 10 engineered features | Group-aware nested RandomizedSearchCV (60 iter) | Classical ML |
| ResNet1D | Raw 1 Hz signal (1800 samples) | Grouped multi-split search (36 configs, 3 grouped tuning folds) | Deep Learning |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FINAL MODEL COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════

all_summaries = [lr_summary, svm_summary, rf_summary, resnet_summary]

comparison_df = pd.DataFrame(all_summaries)
comparison_df = comparison_df.set_index('model')

column_order = [
    'accuracy_mean', 'accuracy_std',
    'balanced_accuracy_mean', 'balanced_accuracy_std',
    'macro_f1_mean', 'macro_f1_std',
    'normal_recall_mean', 'mild_recall_mean', 'severe_recall_mean',
    'normal_recall_std', 'mild_recall_std', 'severe_recall_std',
]
comparison_df = comparison_df[[c for c in column_order if c in comparison_df.columns]]

print("\n" + "=" * 90)
print("MODEL COMPARISON — Record-Step Level (group-aware tuned baselines, mean ± std across folds)")
print("=" * 90)

display_rows = []
for model_name in comparison_df.index:
    row = comparison_df.loc[model_name]
    display_rows.append({
        'Model': model_name,
        'Accuracy': f"{row['accuracy_mean']:.4f} ± {row['accuracy_std']:.4f}",
        'Bal. Accuracy': f"{row['balanced_accuracy_mean']:.4f} ± {row['balanced_accuracy_std']:.4f}",
        'Macro F1': f"{row['macro_f1_mean']:.4f} ± {row['macro_f1_std']:.4f}",
        'Normal Recall': f"{row['normal_recall_mean']:.4f}",
        'Mild Recall': f"{row['mild_recall_mean']:.4f}",
        'Severe Recall': f"{row['severe_recall_mean']:.4f}",
    })

display_df = pd.DataFrame(display_rows).set_index('Model')
print(display_df.to_string())

comparison_csv_path = OUTPUT_DIR / "model_comparison.csv"
comparison_df.to_csv(comparison_csv_path)
print(f"\nComparison table saved to {comparison_csv_path}")

display_csv_path = OUTPUT_DIR / "model_comparison_formatted.csv"
display_df.to_csv(display_csv_path)
print(f"Formatted table saved to {display_csv_path}")

all_fold_detail = []
for model_name, results in [
    ("Logistic Regression", lr_results),
    ("SVM", svm_results),
    ("Random Forest", rf_results),
    ("ResNet1D (grouped-tuned)", resnet_results),
]:
    for r in results:
        all_fold_detail.append({
            'model': model_name,
            'fold': r['fold'],
            'accuracy': r['accuracy'],
            'balanced_accuracy': r['balanced_accuracy'],
            'macro_f1': r['macro_f1'],
            'recall_Normal': r['recall_Normal'],
            'recall_Mild': r['recall_Mild'],
            'recall_Severe': r['recall_Severe'],
        })

fold_detail_df = pd.DataFrame(all_fold_detail)
fold_detail_csv_path = OUTPUT_DIR / "model_comparison_per_fold.csv"
fold_detail_df.to_csv(fold_detail_csv_path, index=False)
print(f"Per-fold detail saved to {fold_detail_csv_path}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

model_names = comparison_df.index.tolist()
x_pos = np.arange(len(model_names))

axes[0].bar(
    x_pos,
    comparison_df['accuracy_mean'],
    yerr=comparison_df['accuracy_std'],
    capsize=4,
    color='steelblue',
    edgecolor='black',
    alpha=0.8,
)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy (mean ± std)')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(
    x_pos,
    comparison_df['balanced_accuracy_mean'],
    yerr=comparison_df['balanced_accuracy_std'],
    capsize=4,
    color='darkorange',
    edgecolor='black',
    alpha=0.8,
)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
axes[1].set_ylabel('Balanced Accuracy')
axes[1].set_title('Balanced Accuracy (mean ± std)')
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(
    x_pos,
    comparison_df['macro_f1_mean'],
    yerr=comparison_df['macro_f1_std'],
    capsize=4,
    color='seagreen',
    edgecolor='black',
    alpha=0.8,
)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
axes[2].set_ylabel('Macro F1')
axes[2].set_title('Macro F1 (mean ± std)')
axes[2].grid(axis='y', alpha=0.3)

fig.suptitle('Model Comparison — Record-Step Level Metrics (group-aware tuned baselines)', fontsize=14, y=1.02)
fig.tight_layout()
comparison_plot_path = OUTPUT_DIR / "model_comparison_barplot.png"
fig.savefig(comparison_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Comparison bar plot saved to {comparison_plot_path}")

fig2, ax2 = plt.subplots(figsize=(12, 6))
bar_width = 0.15
class_colors = {'Normal': '#2ca02c', 'Mild': '#ff7f0e', 'Severe': '#d62728'}

for i, cls_name in enumerate(CLASS_NAMES):
    col = f'{cls_name.lower()}_recall_mean'
    offset = (i - 1) * bar_width
    ax2.bar(
        x_pos + offset,
        comparison_df[col],
        width=bar_width,
        label=f'{cls_name} Recall',
        color=class_colors[cls_name],
        edgecolor='black',
        alpha=0.85,
    )

ax2.set_xticks(x_pos)
ax2.set_xticklabels(model_names, rotation=30, ha='right', fontsize=10)
ax2.set_ylabel('Recall')
ax2.set_title('Per-Class Recall by Model (group-aware tuned baselines)')
ax2.legend(frameon=False)
ax2.grid(axis='y', alpha=0.3)
fig2.tight_layout()
recall_plot_path = OUTPUT_DIR / "model_comparison_recall.png"
fig2.savefig(recall_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Per-class recall plot saved to {recall_plot_path}")

print("\n" + "=" * 90)
print("MODEL COMPARISON COMPLETE")
print("=" * 90)

NameError: name 'lr_summary' is not defined